# Number provenance for manuscript v57 (`manuscript/NUMBERS_v57.md`)

Checks the FINAL manuscript `manuscript/scJDO_manuscript_v57.md` (not edited). For every numeric claim in the
Abstract, "Changes from v56", Results, Discussion, Methods, figure legends and the Supplementary tables list, this
notebook records the exact quoted text and recomputes the value from the committed result files. The scJDO v1 (v56)
analyses are not part of v57 (Results 8 is a pointer to version 56 of the preprint); statements that cite v56 by
version (e.g. "reported in version 56 of this preprint", "in v56, …") are not v57 numbers and are not checked.

Rules (fixed before comparison):
* every quoted snippet must occur verbatim in the named section of the final manuscript (asserted);
* a quoted decimal is compared after rounding the recomputed value to the same number of decimals (half away from
  zero); a quoted range "a–b" (or "a to b") is compared with the recomputed min–max over the stated filter;
  percentages are compared on the % scale;
* "≈ 0" for correlations / cosines means |x| < 0.1 for every row; "≈ 1.0" means every value rounds to 1.0;
* "under X" / "< X" / "≤ X" claims are checked as inequalities (≤ after rounding at the quoted precision);
* values read from the text of a committed result report (RESULT_*.md) count as recomputed (the report is the
  committed result file); numbers that exist only in review notes are NOT-RECOMPUTABLE;
* configuration constants (hyper-parameters, preprocessing choices, thresholds) are NOT-RECOMPUTABLE unless a
  committed result file records them; S22 and S18d numbers are recomputed from the merged result files
  (`benchmarks/real_flow/s22/`, `benchmarks/perturb_dose/`);
* a coverage audit lists every numeric token in the manuscript sections that is not inside a checked snippet and not
  on the explicit list of non-claims (identifiers such as seeds, callouts, dates, accessions, statements citing v56
  by version).

The notebook also checks internal consistency of the final manuscript: figure / table / note callouts against the
figure files and legends, citation numbers against the reference list, and numbers that appear in both the main
text and a legend.

In [1]:
from _style import *
from _style import _rel
import json as _json, re
from decimal import Decimal
from scipy.stats import spearmanr

S15 = S15_DIR   # benchmarks/synthetic_multimodal/s15 (merged)
MS_PATH = "manuscript/scJDO_manuscript_v57.md"
prov = Provenance("NUMBERS", notebook="notebooks/numbers_v57.ipynb", kind="report")
MS = prov.use(MS_PATH, "all", "final manuscript").read_text()
B = "benchmarks/synthetic_multimodal"
F = {}
def load(key, path, task, reader="csv"):
    p = prov.use(path, "numbers", task)
    F[key] = (_rel(p), task)
    if reader == "csv":
        return pd.read_csv(p)
    if reader == "json":
        return _json.loads(p.read_text())
    return p.read_text()

# S08 / S09 / S19
cmp3 = load("cmp3", f"{B}/v3_calibration/compare_v3.csv", "S09")
gates3 = load("gates3", f"{B}/v3_calibration/gates_v3.json", "S09", reader="json")
cmp8 = load("cmp8", "benchmarks/real_reference/compare.csv", "S08")
stats = load("stats", "benchmarks/real_reference/stats.csv", "S08")
dynr = load("dynr", "benchmarks/dyngen/realism.csv", "S19")
# S10 / S12 / S13
ops = load("ops", f"{B}/s10/operators.csv", "S10")
byday = load("byday", f"{B}/s10/by_day.csv", "S10")
diff = load("diff", f"{B}/s10/diffusion.csv", "S10")
dsum = load("dsum", f"{B}/s10/diffusion_summary.json", "S10", reader="json")
s12 = load("s12", f"{B}/s12/step0.csv", "S12")
s12fa = load("s12fa", f"{B}/s12/step0_fa_diagnosis.csv", "S12")
s13 = load("s13", f"{B}/s13/auc.csv", "S13")
# S14 / S14b / S14c
test = load("test", f"{B}/s14/test.csv", "S14")
dev = load("dev", f"{B}/s14/dev_auc_by_k.csv", "S14")
gates14 = load("gates14", f"{B}/s14/gates.json", "S14", reader="json")
choosek = load("choosek", f"{B}/s14/choose_k.csv", "S14")
ksel = load("ksel", f"{B}/s14/k_selection_r1.json", "S14", reader="json")
selcfg = load("selcfg", f"{B}/s14/selected_configuration.json", "S14", reader="json")
add14 = load("add14", f"{B}/s14/addendum_lineage.json", "S14 addendum (post hoc)", reader="json")
elig = load("elig", f"{B}/s14c/g2_transition_eligibility.csv", "S14b/S14c")
eligb = load("eligb", f"{B}/s14b/g2_transition_eligibility.csv", "S14b")
g14c = load("g14c", f"{B}/s14c/gates.json", "S14c", reader="json")
md14c = load("md14c", f"{B}/s14c/RESULT_S14c.md", "S14c", reader="text")
md14b = load("md14b", f"{B}/s14b/RESULT_S14b.md", "S14b", reader="text")
# S15
jref = load("jref", S15 / "jref_diagnosis.csv", "S15")
s15md = load("s15md", S15 / "RESULT_S15.md", "S15", reader="text")
s15g = load("s15g", S15 / "checkpoints/finalise.json", "S15", reader="json")
assert "Complete:" in s15md.split("## Status", 1)[1].split("##", 1)[0], "S15 not complete"
rr = load("rr", S15 / "regime_recovery.csv", "S15")
stab = load("stab", S15 / "stability.csv", "S15")
arch = load("arch", S15 / "archetypes.csv", "S15")
# dyngen
dsc = load("dsc", "benchmarks/dyngen/s21b/scores.csv", "S21b")
dpa = load("dpa", "benchmarks/dyngen/s21b/part_a_T.csv", "S21b")
dmd = load("dmd", "benchmarks/dyngen/s21b/RESULT_S21b.md", "S21b", reader="text")
dman = load("dman", "benchmarks/dyngen/MANIFEST.csv", "S21b")
d21 = load("d21", "benchmarks/dyngen/s21/scores.csv", "S21")
# S11 / S20
va = load("va", "benchmarks/real_flow/velocity_agreement.csv", "S11")
oa = load("oa", "benchmarks/real_flow/operator_agreement.csv", "S11")
nu = load("nu", "benchmarks/real_flow/null.csv", "S11")
bct = load("bct", "benchmarks/real_flow/by_cell_type.csv", "S11")
s11md = load("s11md", "benchmarks/real_flow/RESULT_S11.md", "S11", reader="text")
agr = load("agr", "benchmarks/scaling/agreement.csv", "S20")
scal = load("scal", "benchmarks/scaling/scaling.csv", "S20")
full = load("full", "benchmarks/scaling/larry_full_lambda1.csv", "S20")
s20md = load("s20md", "benchmarks/scaling/RESULT_S20.md", "S20", reader="text")
# S18 / S18b
cap = load("cap", "benchmarks/perturb_k562/capture.csv", "S18")
eff = load("eff", "benchmarks/perturb_k562/effect_tests.json", "S18", reader="json")
pemb = load("pemb", "benchmarks/perturb_k562/embedding.json", "S18", reader="json")
s18md = load("s18md", "benchmarks/perturb_k562/RESULT_S18.md", "S18", reader="text")
h1 = load("h1", "benchmarks/perturb_k562_occupancy/h1.csv", "S18b")
h2 = load("h2", "benchmarks/perturb_k562_occupancy/h2.json", "S18b", reader="json")
s18bmd = load("s18bmd", "benchmarks/perturb_k562_occupancy/RESULT_S18b.md", "S18b", reader="text")
# S22 / S18d (merged result bundles)
s22 = load("s22", "benchmarks/real_flow/s22/scores.csv", "S22")
s22sum = load("s22sum", "benchmarks/real_flow/s22/summary.json", "S22", reader="json")
s22pre = load("s22pre", "benchmarks/real_flow/s22/preregistration.json", "S22", reader="json")
s22fit = load("s22fit", "benchmarks/real_flow/s22/fit_seed0.json", "S22", reader="json")
s22cl = load("s22cl", "benchmarks/real_flow/s22/clones.csv", "S22")
s22md = load("s22md", "benchmarks/real_flow/s22/RESULT_S22.md", "S22", reader="text")
_p = prov.use("benchmarks/real_flow/s22/cells_seed0.npz", "numbers", "S22"); F["s22cells"] = (_rel(_p), "S22")
S22_DIM = int(np.load(_p)["eigenvalues"].shape[1])
_p = prov.use("benchmarks/real_flow/s22/permutation_nulls_seed0.npz", "numbers", "S22"); F["s22null"] = (_rel(_p), "S22")
S22_NPERM = {int(v_.shape[0]) for v_ in np.load(_p).values()}
dg = load("dg", "benchmarks/perturb_dose/gates.json", "S18d", reader="json")
dcap = {(s_, n_): load(f"dcap_{s_}_{n_}", f"benchmarks/perturb_dose/capture_{s_}_{n_}.csv", "S18d") for s_ in ["rpe1", "k562"] for n_ in ["N1", "N2"]}
dcj = {(s_, n_): load(f"dcj_{s_}_{n_}", f"benchmarks/perturb_dose/capture_{s_}_{n_}.json", "S18d", reader="json") for s_ in ["rpe1", "k562"] for n_ in ["N1", "N2"]}

syn = test[test.kind == "synthetic"]; ref = syn[syn.configuration_id == "reference"]; sel = syn[syn.configuration_id == "selected"]
larry = test[test.kind == "larry"].set_index("configuration_id")
def gate_json(name):
    g = s15g["gates"][name]
    assert (re.search(rf"\*\*{name}\*\*: (PASS|FAIL)", s15md).group(1) == "PASS") == g["pass"], name
    return g["by_view"]
A1, A2, A3 = gate_json("gate_A1_saddle_archetype"), gate_json("gate_A2_nmi"), gate_json("gate_A3_stability")
TESTS = ["v3_transition_s2", "v3_transition_s3", "v3_transition_s4"]

## Sections of the final manuscript

In [2]:
body, refs_txt = MS.split("\n## References\n", 1)
main_txt, leg_txt = body.split("\n## Figure legends\n", 1)
def _heads(txt):
    out, cur, buf = {}, None, []
    for ln in txt.split("\n"):
        m = re.match(r"^(#{2,4}) (.*)$", ln)
        if m:
            if cur is not None:
                out[cur] = "\n".join(buf)
            cur, buf = m.group(2).strip(), []
        else:
            buf.append(ln)
    out[cur] = "\n".join(buf)
    return out
H = _heads(main_txt)
SEC = {"Abstract": H["Abstract"], "Changes": H["Changes from v56"]}
for k, v in H.items():
    m = re.match(r"^(\d)\. ", k or "")
    if m:
        SEC[f"Results {m.group(1)}"] = v
_disc = main_txt.split("\n## Discussion\n", 1)[1].split("\n## Methods\n", 1)[0]
SEC["Discussion"] = _disc
_meth = main_txt.split("\n## Methods\n", 1)[1].split("\n## Data Availability\n", 1)[0]
assert "carried over from v56" not in _meth, "unexpected v56 Methods section in the final manuscript"
for k, v in _heads("## x\n" + _meth).items():
    if k != "x":
        SEC[f"Methods: {k}"] = v
for m in re.finditer(r"^\*\*(Figure (\d+)|Supplementary Figure (S\d+)) \|.*$", leg_txt, re.M):
    SEC[f"Legend Fig {m.group(2) or m.group(3)}"] = m.group(0)
SEC["Supplementary tables"] = leg_txt.split("### Supplementary tables", 1)[1]
print(len(SEC), "sections:", list(SEC))

41 sections: ['Abstract', 'Changes', 'Results 1', 'Results 2', 'Results 3', 'Results 4', 'Results 5', 'Results 6', 'Results 7', 'Results 8', 'Discussion', 'Methods: Embedding for flow operators', 'Methods: Time-course velocity', 'Methods: Drift and flow operator', 'Methods: Score operator', 'Methods: Descriptors and scoring', 'Methods: Calibrated synthetic generator', 'Methods: Held-out test of the flow operator (S14)', 'Methods: Transition eligibility (S14b, S14c)', 'Methods: Independent simulator', 'Methods: LARRY', 'Methods: Scaling (S20)', 'Methods: Archetypes (S15)', 'Methods: Perturbation tests (S18, S18b, S18d)', 'Methods: Clonal fate coherence (S22)', 'Methods: Diffusion (S10)', 'Methods: Pre-registration and review', 'Methods: Statistics', 'Methods: Software', 'Legend Fig 1', 'Legend Fig 2', 'Legend Fig 3', 'Legend Fig 4', 'Legend Fig 5', 'Legend Fig 6', 'Legend Fig 7', 'Legend Fig S1', 'Legend Fig S2', 'Legend Fig S3', 'Legend Fig S4', 'Supplementary tables']


In [3]:
def ndec(s):
    s = s.strip().lstrip("+").replace("−", "-")
    return len(s.split(".")[1]) if "." in s else 0

def num(s):
    return float(s.strip().lstrip("+").replace("−", "-").replace(",", ""))

def fmt(x, nd):
    return f2(x, nd) if nd > 0 else str(int(Decimal(repr(float(x))).quantize(Decimal(1), rounding="ROUND_HALF_UP")))

def split_range(e):
    return [t.strip() for t in (e.split(" to ") if " to " in e else e.split("–"))]

ROWS = []
def row(section, quoted, task, src, filt, kind, recompute=None, expect=None, note=""):
    """kind: value | range | values | approx0 | approx1 | lt | le | ge | bool | count | nr"""
    assert section in SEC, section
    assert quoted in SEC[section], f"quoted text not found verbatim in {section}: {quoted!r}"
    srcs = "; ".join(F[s][0] if s in F else s for s in (src if isinstance(src, (list, tuple)) else [src])) if src else "—"
    if kind == "nr":
        ROWS.append(dict(section=section, quoted=quoted, recomputed="—", source=srcs, filter=filt, task=task,
                         status="NOT-RECOMPUTABLE", note=note)); return
    x = recompute()
    if kind == "value":           # expect: quoted number string (may carry %)
        pct = expect.endswith("%"); e = expect.rstrip("%").strip(); nd = ndec(e)
        xv = x * 100 if pct else x
        r = fmt(xv, nd) + ("%" if pct else ""); ok = fmt(xv, nd) == fmt(num(e), nd)
    elif kind == "range":         # expect: "a–b" or "a to b"
        pct = expect.endswith("%"); a, b = split_range(expect.rstrip("%"))
        lo, hi = (min(x), max(x)); lo, hi = (lo * 100, hi * 100) if pct else (lo, hi)
        r = f"{fmt(lo, ndec(a))}–{fmt(hi, ndec(b))}" + ("%" if pct else "")
        ok = fmt(lo, ndec(a)) == fmt(num(a), ndec(a)) and fmt(hi, ndec(b)) == fmt(num(b), ndec(b))
    elif kind == "values":        # expect: list of quoted strings, x list of numbers (same order)
        parts = [fmt(xi, ndec(ei)) for xi, ei in zip(x, expect)]
        assert len(x) == len(expect)
        r = ", ".join(parts); ok = all(p == fmt(num(e), ndec(e)) for p, e in zip(parts, expect))
    elif kind == "approx0":
        xs = np.atleast_1d(x); r = f"max |x| = {f2(np.max(np.abs(xs)), 3)} (n = {len(xs)})"; ok = bool(np.all(np.abs(xs) < 0.1))
    elif kind == "approx1":
        xs = np.atleast_1d(x); r = f"{f2(np.min(xs), 4)}–{f2(np.max(xs), 4)} (n = {len(xs)})"; ok = all(f2(v, 1) == "1.0" for v in xs)
    elif kind == "lt":            # expect: bound (float); x value
        r = f"{f2(x, 2)} (< {expect})"; ok = x < expect
    elif kind in ("le", "ge"):    # expect: quoted bound string; compared at the quoted precision
        nd = ndec(expect); xr = num(fmt(x, nd))
        r = f"{fmt(x, nd)} ({'≤' if kind == 'le' else '≥'} {expect})"; ok = xr <= num(expect) if kind == "le" else xr >= num(expect)
    elif kind in ("bool", "count"):
        r = str(x); ok = (x == expect)
    else:
        raise ValueError(kind)
    ROWS.append(dict(section=section, quoted=quoted, recomputed=r, source=srcs, filter=filt, task=task,
                     status="MATCH" if ok else "MISMATCH", note=note))

def nr(section, quoted, task, why, src=None):
    row(section, quoted, task, src, "—", "nr", note=why)

CFG = "configuration constant (not recorded in a committed result file)"
auc = lambda d: d.auc_lam1.values
OPS = ops[(ops.method == "pca") & (ops.truth_class == "saddle")]
opauc = lambda op, pr="v3_default": OPS[(OPS.operator == op) & (OPS.preset == pr)].auc_saddle_vs_committed.values
T = lambda s: s.astype(str).eq("True")

## Abstract and Changes from v56

In [4]:
S = "Abstract"
row(S, "calibrated to six real datasets", "S08", "stats", "kind == real; distinct dataset", "count",
    lambda: int(stats[stats.kind == "real"].dataset.nunique()), 6)
row(S, "(AUC 0.89–1.00)", "S14", "test", "kind synthetic, configuration_id reference, k 10/15/20, all test presets with a fork", "range",
    lambda: ref.auc_lam1.dropna().values, "0.89–1.00")
row(S, "(Spearman 0.56, permutation p < 0.001; 0.50 after removing the mean shift)", "S14", ["test", "add14"], "larry reference lineage_spearman; addendum drift_pooled p_one_sided (le 0.001), drift_centred spearman", "values",
    lambda: [larry.loc["reference", "lineage_spearman"], add14["drift_pooled"]["p_one_sided"], add14["drift_centred"]["spearman"]], ["0.56", "0.001", "0.50"])
row(S, "95 % CI 0.66–0.73", "S14 addendum", "add14", "heldout_energy ci95 flow", "values",
    lambda: add14["heldout_energy"]["ci95"]["flow"], ["0.66", "0.73"])
row(S, "energy distance 0.69, 95 % CI 0.66–0.73, vs 0.87 and 0.88", "S14", "test", "kind larry, reference: heldout_flow / mean_displacement / no_movement energy", "values",
    lambda: [larry.loc["reference", c] for c in ["heldout_flow_energy", "heldout_mean_displacement_energy", "heldout_no_movement_energy"]], ["0.69", "0.87", "0.88"])
row(S, "(velocity cosine 0.9998)", "S20", "agr", "solver minibatch, n_per_day 8000, median_velocity_cosine", "value",
    lambda: agr[(agr.solver == "minibatch") & (agr.n_per_day == 8000)].median_velocity_cosine.iloc[0], "0.9998")
row(S, "all 130,887 cells", "S20", "full", "sum of n_cells (minibatch, all days and cell types)", "count",
    lambda: int(full.n_cells.sum()), 130887)
mf = scal[(scal.solver == "minibatch") & (scal.n_per_day == "full")]
mf_ot = mf[mf.stage.astype(str).str.contains("ot", case=False)] if "stage" in mf else mf.iloc[:1]
mf_fit = mf[~mf.index.isin(mf_ot.index)]
row(S, "(transport 5 min and 2.5 GB;", "S20", "scal", "minibatch full OT stage: wall/60, max_rss_bytes/1e9", "values",
    lambda: [mf_ot.wall_seconds_external.sum() / 60, mf_ot.max_rss_bytes.max() / 1e9], ["5", "2.5"])
row(S, "drift fit and Jacobians 21 min and 1.3 GB)", "S20", "scal", "minibatch full fit_and_jacobian stage: wall/60, max_rss_bytes/1e9", "values",
    lambda: [mf_fit.wall_seconds_external.sum() / 60, mf_fit.max_rss_bytes.max() / 1e9], ["21", "1.3"])
bif = dsc[(dsc.backbone == "bifurcating") & (dsc.operator == "flow")]
row(S, "detected two of three bifurcations", "S21b", "dsc", "flow; datasets with fork_auc >= 0.85 on all 3 drift seeds (gate F1 threshold)", "count",
    lambda: int((bif.groupby("dataset").fork_auc.min() >= 0.85).sum()), 2)
fl = dsc[dsc.operator == "flow"].copy(); fl["pair"] = fl.dataset.str.extract(r"_(s\d)_T8")[0]
def n_cycles(f=fl):
    n = 0
    for p in ["s0", "s1", "s2"]:
        c = f[(f.backbone == "cycle") & (f.pair == p)].sort_values("drift_seed").median_rotation.values
        l = f[(f.backbone == "linear") & (f.pair == p)].sort_values("drift_seed").median_rotation.values
        n += bool(np.all(c > l))
    return n
row(S, "two of three cycles", "S21b", "dsc", "flow; cycle median_rotation > paired linear on all 3 drift seeds", "count", n_cycles, 2)
lin_all = [float(m.group(1)) for m in re.finditer(r"\| linear_s\d_T8 \| \d \| linear_all \| 5000 \| ([0-9.]+) \|", dmd)]
assert len(lin_all) == 9
lin_pos = lambda: float(max(lin_all + list(fl[fl.backbone == "linear"][["fraction_positive_early", "fraction_positive_committed"]].values.ravel())))
row(S, "never reported instability on linear trajectories", "S21b", ["dsc", "dmd"],
    "linear backbones: fraction Re λ1 > 0 (scores.csv early+committed; RESULT_S21b.md linear_all table, 9 rows)", "count", lin_pos, 0.0)

S = "Changes"
row(S, "the OT flow operator inverts on FA (fork AUC 0.04–0.53)", "S12", "s12", "embedding == fa20 (6 datasets)", "range",
    lambda: auc(s12[s12.embedding == "fa20"]), "0.04–0.53")
row(S, "PCA of log-normalized counts, 10–20 components", "S14", "test", "k values tested in the S14 held-out test", "range",
    lambda: test.k.unique(), "10–20")
row(S, "a generator calibrated to six real datasets", "S08", "stats", "kind == real; distinct dataset", "count",
    lambda: int(stats[stats.kind == "real"].dataset.nunique()), 6)

## Results 1–2

In [5]:
S = "Results 1"
row(S, "in a variance-weighted transport space (PCA of log-normalized counts, 10–20 components)", "S14", "test", "k values tested in the S14 held-out test", "range",
    lambda: test.k.unique(), "10–20")
row(S, "a rotational signal on two of three simulated cycles at eight time points", "S21b", ["dsc", "dpa"],
    "cycles passing (all drift seeds) ; T of the S21b re-sampling", "values", lambda: [n_cycles(), dpa["T"].max()], ["2", "8"])

S = "Results 2"
ids = lambda d: sorted(set(re.match(r"^[A-Z]\d+", s).group(0) for s in d.statistic))
row(S, "We measured 20 statistics (32 sub-statistics) on six real datasets", "S08/S09", ["cmp8", "cmp3", "stats"],
    "distinct statistic IDs (compare.csv); distinct sub-statistics (compare_v3.csv); distinct real datasets (stats.csv)", "values",
    lambda: [len(ids(cmp8)), cmp3.statistic.nunique(), stats[stats.kind == "real"].dataset.nunique()], ["20", "32", "6"], note="sub-statistics counted in compare_v3.csv (the S08 comparison table as extended by S09, plotted in Fig. 2); S08 compare.csv has " + str(cmp8.statistic.nunique()) + ", stats.csv has " + str(stats.statistic.nunique()) + " statistic labels")
z = cmp8[cmp8.statistic == "C5_zero_fraction"]
z3 = cmp3[cmp3.statistic == "C5_zero_fraction"]
z3med = lambda gname: float(np.nanmedian(z3[z3.preset.map(lambda p: "v1" if p == "v1" else p.split("_")[0]) == gname].synthetic_scalar_or_median.astype(float)))
row(S, "the median zero fraction was 36–37 % in v1 and v2 data against 84–96 % in real data", "S08/S09", "cmp3",
    "C5_zero_fraction: median synthetic_scalar_or_median over the v1 datasets and over the v2 datasets (as plotted in Fig. 2a); min–max of the two", "range",
    lambda: [z3med("v1"), z3med("v2")], "36–37%", note=f"v1 {f2(z3med('v1'), 3)}, v2 {f2(z3med('v2'), 3)}")
row(S, "the median zero fraction was 36–37 % in v1 and v2 data against 84–96 % in real data", "S08", "cmp8",
    "C5_zero_fraction: real_range_min – real_range_max", "range", lambda: [z.real_range_min.iloc[0], z.real_range_max.iloc[0]], "84–96%")
rz = gates3["realism"]
row(S, "Against 13 calibration targets fixed in advance it met 6 of 13 on each seed", "S09", "gates3",
    "gates_v3.json realism: n_statistics; min and max met_count_by_seed", "values",
    lambda: [rz["n_statistics"], min(rz["met_count_by_seed"].values()), max(rz["met_count_by_seed"].values())], ["13", "6", "6"])
row(S, "the targets missed on all three seeds were genes detected, the fraction of clones spanning several time points, the unspliced fraction, the share of steady genes and the scVelo recovered fraction",
    "S09", "gates3", "intersection of unmet_by_seed over the 3 seeds", "bool",
    lambda: sorted(set.intersection(*[set(v) for v in rz["unmet_by_seed"].values()])),
    ["C2_genes_detected", "G3_multi_time_clone_fraction", "S1_unspliced_fraction", "S3_steady_share", "S4_recovered_fraction"])
mr = cmp3.assign(m=cmp3.match.astype(str).eq("True")).groupby("preset").m.mean()
row(S, "Its overall match rate was 0.39 (v1 0.35, v2 0.22; Fig. 2c)", "S09", "cmp3", "mean(match) for preset v3_default, v1, v2_default", "values",
    lambda: [mr["v3_default"], mr["v1"], mr["v2_default"]], ["0.39", "0.35", "0.22"])
dmt = dynr.assign(m=dynr.match.astype(str).eq("True"))
row(S, "matched the same statistics at 0.51 (range 0.44–0.60)", "S19", "dynr", "mean(match) over all 9 datasets; per-dataset min–max", "values",
    lambda: [dmt.m.mean(), dmt.groupby("synthetic").m.mean().min(), dmt.groupby("synthetic").m.mean().max()], ["0.51", "0.44", "0.60"])
row(S, "time-course sampling (days 2, 4 and 6)", "S10", "byday", "distinct day values in by_day.csv", "values",
    lambda: sorted(byday.day.unique()), ["2", "4", "6"])

## Results 3 — flow operator from a time course

In [6]:
S = "Results 3"
nr(S, "ε = 0.05 × median squared distance", "S10/S14", CFG + "; RESULT_S15.md 'Frozen configuration' states 'epsilon factor 0.05'")
row(S, "PCA with k = 15", "S14", "ksel", "k_selection_r1.json k_star", "count", lambda: int(ksel["k_star"]) if "k_star" in ksel else int(selcfg["k_star"]), 15)
row(S, "tuned one factor at a time on two development seeds", "S14", "dev", "development seeds per preset in dev_auc_by_k.csv", "count",
    lambda: int(dev.groupby("preset").seed.nunique().max()), 2)
row(S, "AUC 0.89–0.97 across k = 10–20 principal components", "S14", "test", "reference, preset v3_transition (s2–s4), k 10/15/20", "range",
    lambda: auc(ref[ref.preset == "v3_transition"]), "0.89–0.97")
row(S, "AUC 0.89–0.97 across k = 10–20 principal components", "S14", "test", "k values of the reference rows", "range",
    lambda: ref.k.unique(), "10–20")
row(S, "0.97–1.00 on the default, batch and stress presets", "S14", "test", "reference, presets v3_default/v3_batch/v3_stress, k 10/15/20", "range",
    lambda: auc(ref[ref.preset.isin(["v3_default", "v3_batch", "v3_stress"])]), "0.97–1.00")
row(S, "did worse on the same held-out seeds (0.42–0.95)", "S14", "test", "selected, preset v3_transition, k 10/15/20", "range",
    lambda: auc(sel[sel.preset == "v3_transition"]), "0.42–0.95")
row(S, "All four pre-registered gates", "S14", "gates14", "gates G1–G4 in gates.json: number of gates, number failed", "values",
    lambda: [sum(k.startswith("gate_G") for k in gates14), sum(k.startswith("gate_G") and not gates14[k]["pass"] for k in gates14)], ["4", "4"])
row(S, "On the default preset (PCA-20; Fig. 3b), the OT flow operator reached fork AUC 0.97–1.00", "S10", "ops", "PCA, saddle rows, v3_default, J_flow_ot, seeds 0–2", "range",
    lambda: opauc("J_flow_ot"), "0.97–1.00")
row(S, "labeling velocity (0.53–0.89)", "S10", "ops", "PCA, v3_default, J_flow_raw_shared and J_flow_smoothed_shared", "range",
    lambda: np.r_[opauc("J_flow_raw_shared"), opauc("J_flow_smoothed_shared")], "0.53–0.89")
row(S, "scVelo velocity (0.27–0.51)", "S10", "ops", "PCA, v3_default, J_flow_scvelo", "range", lambda: opauc("J_flow_scvelo"), "0.27–0.51")
row(S, "ranked committed cells above saddle cells (0.16–0.31)", "S10", "ops", "PCA, v3_default, J_score", "range", lambda: opauc("J_score"), "0.16–0.31")
for e, q in [("pca10", "PCA-10 (0.88–1.00)"), ("pca20", "PCA-20 (0.82–1.00)"), ("pca50", "PCA-50 (0.56–0.97)"),
             ("scvi10", "scVI-10 [45] (0.31–0.96)"), ("fa20", "Factor Analysis (FA-20; 0.04–0.53, inverted)")]:
    row(S, q, "S12", "s12", f"embedding == {e} (v3_default and v3_transition s0–2)", "range",
        lambda e=e: auc(s12[s12.embedding == e]), re.search(r"(\d\.\d\d–\d\.\d\d)", q).group(1))
row(S, "OT velocities agree between PCA and FA in gene space (0.52–0.56)", "S12", "s12fa", "step0_fa_diagnosis b_gene_space_cos_median, 6 datasets", "range",
    lambda: s12fa.b_gene_space_cos_median.values, "0.52–0.56")
t13 = s13[s13.preset == "v3_transition"]
a13 = lambda e, v: auc(t13[(t13.embedding == e) & (t13.variant == v)])
row(S, "whitening the embedding destroyed the signal (PCA-20 0.37–0.58)", "S13", "s13", "v3_transition s0–2, pca20, whitened_full", "range", lambda: a13("pca20", "whitened_full"), "0.37–0.58")
row(S, "whitening only the drift fit preserved it (PCA-10 0.79–0.96)", "S13", "s13", "v3_transition s0–2, pca10, raw_ot_whitened_drift", "range", lambda: a13("pca10", "raw_ot_whitened_drift"), "0.79–0.96")
row(S, "whitening only the OT cost degraded it (PCA-10 0.53–0.79; PCA-20 0.28–0.43)", "S13", "s13", "v3_transition s0–2, whitened_ot_raw_drift, pca10 / pca20", "values",
    lambda: [a13("pca10", "whitened_ot_raw_drift").min(), a13("pca10", "whitened_ot_raw_drift").max(), a13("pca20", "whitened_ot_raw_drift").min(), a13("pca20", "whitened_ot_raw_drift").max()],
    ["0.53", "0.79", "0.28", "0.43"])
row(S, "The median per-cell relative error is ≈ 1.0 on every held-out dataset (1.000–1.005)", "S14", "test", "jref_relative_frobenius_median, all synthetic test rows", "approx1",
    lambda: syn.jref_relative_frobenius_median.dropna().values)
row(S, "The median per-cell relative error is ≈ 1.0 on every held-out dataset (1.000–1.005)", "S14", "test", "jref_relative_frobenius_median, all synthetic test rows: min–max", "range",
    lambda: syn.jref_relative_frobenius_median.dropna().values, "1.000–1.005")
row(S, "The flow operator's norm is 2–7 % of the reference's", "S15", "jref", "median_norm_J_over_norm_Jref, all 8 rows (4 datasets × J_ref kNN 15/30)", "range",
    lambda: jref.median_norm_J_over_norm_Jref.values, "2–7%")
row(S, "optimal rescaling does not reduce the error", "S15", "jref", "median_relative_error_cstar_J ≈ 1.0, all rows", "approx1", lambda: jref.median_relative_error_cstar_J.values)
row(S, "the median per-cell cosine is ≈ 0 (−0.002 to 0.008)", "S15", "jref", "median_per_cell_cosine, all rows", "approx0", lambda: jref.median_per_cell_cosine.values)
row(S, "the median per-cell cosine is ≈ 0 (−0.002 to 0.008)", "S15", "jref", "median_per_cell_cosine, all rows: min–max", "range", lambda: jref.median_per_cell_cosine.values, "−0.002 to 0.008")
row(S, "the per-cell λ₁ Spearman correlation is −0.107 to −0.005 across all four datasets", "S15", "jref", "lambda1_spearman min / max, all 8 rows", "range",
    lambda: jref.lambda1_spearman.values, "−0.107 to −0.005")
row(S, "−0.107 and −0.086 on v3_default s2", "S15", "jref", "lambda1_spearman, v3_default_s2 rows (min, max)", "values",
    lambda: sorted(jref[jref.dataset == "v3_default_s2"].lambda1_spearman.values), ["−0.107", "−0.086"])
row(S, "(−0.018 to −0.005 on the three transition seeds;", "S15", "jref", "lambda1_spearman min / max, v3_transition_s2–s4 rows (J_ref kNN 15 and 30)", "range",
    lambda: jref[jref.dataset.str.startswith("v3_transition")].lambda1_spearman.values, "−0.018 to −0.005")
row(S, "with median per-cell cosine ≈ 1.000 against *J_ref*", "S15", "jref", "basis_jref_fd_median_per_cell_cosine (min over rows)", "value",
    lambda: jref.basis_jref_fd_median_per_cell_cosine.min(), "1.000")
r15 = ref[(ref.k == 15)]
row(S, "on the three held-out transition seeds and three no-fork seeds", "S14", "test", "reference k 15: number of v3_transition and v3_nofork datasets", "values",
    lambda: [r15[r15.preset == "v3_transition"].dataset.nunique(), r15[r15.preset == "v3_nofork"].dataset.nunique()], ["3", "3"])
row(S, "separated all transitions (1.91–2.69)", "S14", "test", "reference, k 15, v3_transition: transition_peak_to_median", "range",
    lambda: r15[r15.preset == "v3_transition"].transition_peak_to_median.values, "1.91–2.69")
row(S, "from all no-forks (0.58–1.44)", "S14", "test", "reference, k 15, v3_nofork: transition_peak_to_median", "range",
    lambda: r15[r15.preset == "v3_nofork"].transition_peak_to_median.values, "0.58–1.44")
row(S, "with timing offsets of 0, 1 and 5 bins", "S14", "test", "reference, k 15, v3_transition: sorted transition_abs_offset", "values",
    lambda: sorted(r15[r15.preset == "v3_transition"].transition_abs_offset.values), ["0", "1", "5"])
el = elig.copy()
el["bins"] = el.bin_counts.map(lambda s_: _json.loads(s_) if isinstance(s_, str) else None)
el["min_bin"] = el.bins.map(lambda b_: min(b_) if b_ else np.nan)
done = el.generation_status.eq("completed"); eligible = T(el.eligible)
only_bins = done & ~eligible & ~T(el.all_bins_at_least_50_cells) & T(el.true_peak_bin_2_to_9) & T(el.bootstrap_stability_at_least_90pct) & T(el.strict_root_provenance_complete)
row(S, "Two pre-registered confirmatory tests were designed to require four transition datasets", "S14c", "g14c", "gates.json required_eligible_seeds", "count",
    lambda: int(g14c["required_eligible_seeds"]), 4)
row(S, "at least 50 cells in every one of 12 bins", "S14b/S14c", "elig",
    "number of bins per profile; rule recomputed: (min bin_counts >= 50) == all_bins_at_least_50_cells for every assessed seed", "values",
    lambda: [el[done].bins.map(len).max(), int(all((el[done].min_bin >= 50) == T(el[done].all_bins_at_least_50_cells)))], ["12", "1"],
    note="second value 1 = threshold 50 reproduces the committed flag on all assessed seeds")
row(S, "the same peak bin in at least 90 % of 200 bootstrap resamples", "S14b/S14c", "elig",
    "rule recomputed: (bootstrap_stability >= 0.90) == flag; bootstrap_replicates", "values",
    lambda: [int(all((el[done].bootstrap_stability >= 0.9) == T(el[done].bootstrap_stability_at_least_90pct))), el[done].bootstrap_replicates.max()], ["1", "200"],
    note="first value 1 = threshold 0.90 reproduces the committed flag")
b14 = el[el.seed <= 16]
row(S, "only 2 of 9 assessed seeds were eligible", "S14b", "elig", "S14b reserve s5–s16: eligible / completed generations", "values",
    lambda: [int(T(b14.eligible).sum()), int(b14.generation_status.eq("completed").sum())], ["2", "9"])
row(S, "three further seeds failed in the generator", "S14b", "elig", "S14b reserve s5–s16: generation_status != completed", "count",
    lambda: int((~b14.generation_status.eq("completed")).sum()), 3)
c14 = el[el.seed > 16]
row(S, "an extended reserve of 24 seeds yielded no further eligible dataset", "S14c", "elig", "S14c seeds (> 16): count; eligible", "values",
    lambda: [len(c14), int(T(c14.eligible).sum())], ["24", "0"])
row(S, "14 seeds stopped at the memory guard during generation and none of the 10 assessed was eligible", "S14c", "elig",
    "S14c: guard_stop count; completed count; eligible among completed", "values",
    lambda: [int(c14.generation_status.eq("guard_stop").sum()), int(c14.generation_status.eq("completed").sum()), int(T(c14[c14.generation_status.eq("completed")].eligible).sum())], ["14", "10", "0"])
row(S, "the three seeds with stable interior peaks (bootstrap stability 0.955–1.000) failed only the bin-occupancy rule", "S14b/S14c", "elig",
    "seeds failing only all_bins_at_least_50_cells: count; bootstrap_stability min–max", "values",
    lambda: [int(only_bins.sum()), el[only_bins].bootstrap_stability.min(), el[only_bins].bootstrap_stability.max()], ["3", "0.955", "1.000"],
    note="seeds " + ", ".join(str(s_) for s_ in el[only_bins].seed))
row(S, "with minimum bins of 42–46 cells", "S14b/S14c", "elig", "seeds failing only the bin rule: min(bin_counts)", "range",
    lambda: el[only_bins].min_bin.values, "42–46")
row(S, "the true critical peak was usually flat on the 12-bin scale", "S14b", "elig", "number of bins in true_profile (assessed seeds)", "count",
    lambda: int(el[done].true_profile.map(lambda s_: len(_json.loads(s_))).max()), 12)
nr(S, "set by about 180 day-2 cells", "S14c (review)", "value only in review note coord/orient/reviews/S14c.md ('Day-2 has ≈ 180 cells'); no committed result column")

## Results 4 — dyngen

In [7]:
S = "Results 4"
t8 = dman[dman.key.str.endswith("_T8")]
row(S, "On dyngen data (5,000 cells, 1,000 genes, eight synchronized sampling times", "S21b", ["dman", "dpa"],
    "MANIFEST.csv _T8 rows: cells, genes (unique); part_a_T.csv T (unique)", "values",
    lambda: [t8.cells.unique()[0] if t8.cells.nunique() == 1 else np.nan, t8.genes.unique()[0] if t8.genes.nunique() == 1 else np.nan,
             dpa["T"].unique()[0] if dpa["T"].nunique() == 1 else np.nan], ["5000", "1000", "8"])
fa = bif.groupby("dataset").fork_auc
row(S, "Flow-operator fork AUC was 1.00, 0.92–0.99 and 0.31–0.38 (inverted)", "S21b", "dsc", "flow, bifurcating s10 / s16 / s18: min, min, max, min, max over drift seeds", "values",
    lambda: [fa.min()["bifurcating_v2_s10_T8"], fa.min()["bifurcating_v2_s16_T8"], fa.max()["bifurcating_v2_s16_T8"], fa.min()["bifurcating_v2_s18_T8"], fa.max()["bifurcating_v2_s18_T8"]],
    ["1.00", "0.92", "0.99", "0.31", "0.38"], note="s10 all three seeds = 1.000")
row(S, "consistent across three drift-fit seeds", "S21b", "dsc", "number of distinct flow drift seeds", "count", lambda: int(bif.drift_seed.nunique()), 3)
row(S, "the pre-registered gate (≥ 0.85 on every dataset) failed", "S21b", "dsc", "every bifurcating dataset has min fork_auc >= 0.85", "bool",
    lambda: bool((bif.groupby("dataset").fork_auc.min() >= 0.85).all()), False)
scj = dsc[(dsc.backbone == "bifurcating") & (dsc.operator == "score")].set_index("dataset").fork_auc
row(S, "The score operator reached 0.98, 0.58 and 0.85 (Fig. 4a)", "S21b", "dsc", "operator score, bifurcating s10/s16/s18", "values",
    lambda: [scj["bifurcating_v2_s10_T8"], scj["bifurcating_v2_s16_T8"], scj["bifurcating_v2_s18_T8"]], ["0.98", "0.58", "0.85"])
cyc = fl[(fl.backbone == "cycle") & fl.pair.isin(["s1", "s2"])].median_rotation
lin = fl[(fl.backbone == "linear") & fl.pair.isin(["s1", "s2"])].median_rotation
row(S, "for two of three cycles (0.19–0.40 vs 0.07–0.09)", "S21b", "dsc", "cycles passing; flow median_rotation, cycle s1+s2 (6 rows) vs linear s1+s2 (6 rows)", "values",
    lambda: [n_cycles(), cyc.min(), cyc.max(), lin.min(), lin.max()], ["2", "0.19", "0.40", "0.07", "0.09"])
row(S, "but not for the third (0.02 vs 0.08–0.10)", "S21b", "dsc", "flow median_rotation, cycle s0 (3 drift seeds) min/max; linear s0 min/max", "values",
    lambda: [fl[fl.dataset == "cycle_s0_T8"].median_rotation.min(), fl[fl.dataset == "cycle_s0_T8"].median_rotation.max(),
             fl[fl.dataset == "linear_s0_T8"].median_rotation.min(), fl[fl.dataset == "linear_s0_T8"].median_rotation.max()], ["0.02", "0.02", "0.08", "0.10"])
row(S, "No cell had Re λ₁ > 0, under any drift-fit seed (Fig. 4c)", "S21b", ["dsc", "dmd"], "linear backbones, all 5,000 cells (RESULT_S21b.md table) and early/committed regions", "count", lin_pos, 0.0)
t3 = d21[(d21.backbone == "bifurcating") & (d21.operator == "flow")].groupby("dataset").n_fork.first()
row(S, "They placed 22, 0 and 0 cells at the branch point", "S21", "d21", "S21 (T = 3) n_fork per bifurcating dataset (s10, s16, s18)", "values",
    lambda: [t3["bifurcating_v2_s10"], t3["bifurcating_v2_s16"], t3["bifurcating_v2_s18"]], ["22", "0", "0"])
t8n = dsc[(dsc.backbone == "bifurcating") & (dsc.operator == "flow")].groupby("dataset").n_fork.first()
row(S, "(against 329, 746 and 776 at eight times)", "S21b", "dsc", "S21b (T = 8) n_fork per bifurcating dataset (s10, s16, s18)", "values",
    lambda: [t8n["bifurcating_v2_s10_T8"], t8n["bifurcating_v2_s16_T8"], t8n["bifurcating_v2_s18_T8"]], ["329", "746", "776"])
row(S, "so the fork test was evaluable on one dataset only", "S21", "d21", "S21 bifurcating datasets with n_fork > 0", "count", lambda: int((t3 > 0).sum()), 1)
e3 = d21[(d21.backbone == "bifurcating") & (d21.operator == "flow")].fraction_positive_early
row(S, "every early (pre-branch) cell had Re λ₁ > 0", "S21", "d21", "S21 flow, bifurcating: min fraction_positive_early over datasets × seeds", "value", lambda: e3.min(), "1.00")
fe = fl[fl.backbone == "bifurcating"].groupby("dataset").fraction_positive_early
row(S, "this fraction fell to 0.00–0.06 on two of three datasets (1.00 on the third)", "S21b", "dsc",
    "flow fraction_positive_early, s10+s18 (min–max over seeds); s16 (min)", "values",
    lambda: [min(fe.min()["bifurcating_v2_s10_T8"], fe.min()["bifurcating_v2_s18_T8"]), max(fe.max()["bifurcating_v2_s10_T8"], fe.max()["bifurcating_v2_s18_T8"]), fe.min()["bifurcating_v2_s16_T8"]],
    ["0.00", "0.06", "1.00"])

## Results 5 — real data and scale

In [8]:
S = "Results 5"
row(S, "days 2, 4 and 6 [40]; dense analyses use a fixed sample of 4,000 cells per day", "S14", "test", "kind larry n_cells / 3 sampling days", "count",
    lambda: int(larry.loc["reference", "n_cells"] / 3), 4000, note="days 2/4/6 of LARRY are dataset design (not re-derived here)")
vl = va[(va.dataset == "larry") & (va.source_a == "ot") & (va.source_b == "lineage")].velocity_cosine
row(S, "(median cosine 0.69 over 1,321 cells with a lineage velocity)", "S11", "va", "larry, ot vs lineage: median velocity_cosine; number of cells", "values",
    lambda: [vl.median(), len(vl)], ["0.69", "1321"])
row(S, "pre-registered statistic: Spearman 0.56, pooled over the 15 PCA coordinates of 554 cell–clone pairs; maximum over five permutation nulls 0.15", "S14", ["test", "add14"], "larry reference lineage_spearman, k, n pairs (addendum), number of nulls, max null", "values",
    lambda: [larry.loc["reference", "lineage_spearman"], 15, add14["n_cell_barcode_pairs"], len(_json.loads(larry.loc["reference", "lineage_nulls"])), max(_json.loads(larry.loc["reference", "lineage_nulls"]))], ["0.56", "15", "554", "5", "0.15"])
row(S, "Post hoc, 1,000 permutations give a one-sided p < 0.001 (null maximum 0.15)", "S14 addendum", "add14", "drift_pooled: n_permutations, p_one_sided (= 1/1001), null_max", "values",
    lambda: [add14["n_permutations"], add14["drift_pooled"]["p_one_sided"], add14["drift_pooled"]["null_max"]], ["1000", "0.001", "0.15"])
row(S, "a population mean-shift predictor reaches 0.19; after centring every coordinate within each interval, the drift still agrees with clonal displacement (0.50; null maximum 0.04); the OT displacement without a drift reaches 0.54 (0.48 centred); and displacement magnitudes agree weakly (0.24). λ₁ is unrelated to displacement magnitude (−0.04)", "S14 addendum", "add14", "mean_shift, drift_centred (+null max), ot_displacement pooled/centred, magnitude_drift_vs_actual, lambda1_vs_actual_magnitude", "values",
    lambda: [add14["mean_shift_pooled"]["spearman"], add14["drift_centred"]["spearman"], add14["drift_centred"]["null_max"], add14["ot_displacement_pooled"]["spearman"], add14["ot_displacement_centred"]["spearman"], add14["magnitude_drift_vs_actual"], add14["lambda1_vs_actual_magnitude"]], ["0.19", "0.50", "0.04", "0.54", "0.48", "0.24", "−0.04"])
nr(S, "over each 48-h interval", "S14", "sampling gap of LARRY (days 2/4/6); configuration constant")
row(S, "post hoc 95 % bootstrap intervals 0.66–0.73, 0.85–0.89 and 0.83–0.94, with the flow best in all 200 resamples", "S14 addendum", "add14", "heldout_energy ci95 flow, mean_displacement, no_movement; n_boot; frac_boot_flow_best", "values",
    lambda: [*(add14["heldout_energy"]["ci95"][m][i] for m in ("flow", "mean_displacement", "no_movement") for i in (0, 1)), add14["heldout_energy"]["n_boot"], add14["heldout_energy"]["frac_boot_flow_best"] * 200], ["0.66", "0.73", "0.85", "0.89", "0.83", "0.94", "200", "200"])
row(S, "Transport and drift were fitted on days 2 and 4 only (Methods)", "S14 addendum", "add14", "heldout_energy train_days", "values", lambda: add14["heldout_energy"]["train_days"], ["2", "4"])
row(S, "(energy distance 0.69 vs 0.87 and 0.88;", "S14", "test", "kind larry, reference", "values",
    lambda: [larry.loc["reference", c] for c in ["heldout_flow_energy", "heldout_mean_displacement_energy", "heldout_no_movement_energy"]], ["0.69", "0.87", "0.88"])
row(S, "The tuned configuration did not (0.94)", "S14", "test", "kind larry, selected: heldout_flow_energy", "value",
    lambda: larry.loc["selected", "heldout_flow_energy"], "0.94")
st = oa[(oa.dataset == "larry") & (oa.left == "ot") & (oa.right == "ot") & oa.right_estimator.isin(["measured_s1", "measured_s2", "dynamo"])]
row(S, "(λ₁ Spearman 0.92–0.95, including Dynamo's SparseVFC on the same velocity [16])", "S11", "oa",
    "larry, ot vs ot, right_estimator measured_s1 / measured_s2 / dynamo: lambda1_spearman", "range", lambda: st.lambda1_spearman.values, "0.92–0.95")
L = bct[(bct.dataset == "larry") & (bct.operator == "measured")]
def undiff_top():
    out = []
    for s in ["ot", "lineage"]:
        t = L[L.source == s].set_index("cell_type").lambda1_median
        out.append(bool(t["Undifferentiated"] > max(t["Neutrophil"], t["Monocyte"])))
    return all(out)
row(S, "both the OT and the lineage flow operators rank undifferentiated cells above neutrophils and monocytes", "S11", "bct", "larry, operator measured, sources ot and lineage: median λ1", "bool", undiff_top, True)
ind = oa[(oa.dataset == "larry") & (oa.comparison_kind == "independent_sources")].iloc[0]
nl = nu[(nu.dataset == "larry") & (nu.null_kind == "independent_sources") & (nu.left == "ot") & (nu.right == "lineage")]
row(S, "(λ₁ Spearman 0.27, null maximum 0.27; leading-eigenvector |cos| 0.29, null maximum 0.32", "S11", ["oa", "nu"],
    "larry independent_sources ot vs lineage: lambda1_spearman, max null; leading_eigenvector_abs_cosine_median, max null", "values",
    lambda: [ind.lambda1_spearman, nl.lambda1_spearman.max(), ind.leading_eigenvector_abs_cosine_median, nl.leading_eigenvector_abs_cosine_median.max()],
    ["0.27", "0.27", "0.29", "0.32"])
crit = re.findall(r"^\| (?:independent_sources|estimators) \|.*\| (True|False) \|$", s11md, re.M)
row(S, "This pre-registered real-data test met 6 of 14 criteria", "S11", "s11md", "RESULT_S11.md criterion table: rows passing / rows", "values",
    lambda: [crit.count("True"), len(crit)], ["6", "14"])
nr(S, "(4,000-cell batches, each cell in ten batches)", "S20", CFG + " (mini-batch solver setting)")
a8 = agr[(agr.solver == "minibatch") & (agr.n_per_day == 8000)].iloc[0]
row(S, "at 8,000 cells per day (median velocity cosine 0.9998; λ₁ Spearman 0.9999)", "S20", "agr", "minibatch, n_per_day 8000", "values",
    lambda: [a8.n_per_day, a8.median_velocity_cosine, a8.flow_lambda1_spearman], ["8000", "0.9998", "0.9999"])
mo = mf.set_index("stage")
row(S, "all 130,887 LARRY cells in 291 s (2.5 GB)", "S20", ["full", "scal"], "n_cells sum; minibatch full OT wall_seconds_external; max_rss_bytes/1e9", "values",
    lambda: [full.n_cells.sum(), mo.loc["ot", "wall_seconds_external"], mo.loc["ot", "max_rss_bytes"] / 1e9], ["130887", "291", "2.5"])
row(S, "plus drift fitting and Jacobians in 21 min (1.3 GB)", "S20", "scal", "minibatch full fit_and_jacobian: wall/60, max_rss_bytes/1e9", "values",
    lambda: [mo.loc["fit_and_jacobian", "wall_seconds_external"] / 60, mo.loc["fit_and_jacobian", "max_rss_bytes"] / 1e9], ["21", "1.3"])
gb21 = lambda: float(re.search(r"cost matrix is approximately ([0-9.]+) GB", s20md).group(1))
row(S, "dense OT on the full data was not run (cost matrix ≈ 21 GB)", "S20", ["scal", "s20md"], "no dense/full row in scaling.csv; RESULT_S20.md cost-matrix estimate", "values",
    lambda: [int(((scal.solver == "dense") & (scal.n_per_day == "full")).sum()), gb21()], ["0", "21"])
g = full.groupby("cell_type")
ctrho = lambda: spearmanr(g.apply(lambda x: np.average(x.lambda1_median, weights=x.n_cells), include_groups=False), g.s11_lambda1_median.first()).statistic
row(S, "agrees with the 4,000-per-day result (Spearman 0.82 over 11 cell types)", "S20", "full",
    "Spearman over cell types: cell-count-weighted mean of per-day λ1 medians vs s11_lambda1_median; number of cell types", "values",
    lambda: [ctrho(), full.cell_type.nunique()], ["0.82", "11"])
lr = agr[agr.solver.str.startswith("lowrank")]
row(S, "Low-rank OT (moscot, rank 50 or 200 [27])", "S20", "agr", "low-rank solver ranks present", "values",
    lambda: sorted(int(s_.split("_r")[1]) for s_ in lr.solver.unique()), ["50", "200"])
row(S, "(velocity cosine 0.79–0.80; λ₁ Spearman ≈ 0)", "S20", "agr", "lowrank_r50 and lowrank_r200, all n_per_day: median_velocity_cosine", "range",
    lambda: lr.median_velocity_cosine.values, "0.79–0.80")
row(S, "(velocity cosine 0.79–0.80; λ₁ Spearman ≈ 0)", "S20", "agr", "lowrank, all n_per_day: flow_lambda1_spearman", "approx0", lambda: lr.flow_lambda1_spearman.values)
vsc = lambda a, b: va[(va.dataset == "scnt_hema") & (va.source_a == a) & (va.source_b == b)].velocity_cosine.median()
row(S, "(median per-cell cosine: labeling vs Dynamo −0.05, labeling vs scVelo 0.03, scVelo vs Dynamo 0.15; Fig. 5b)", "S11", "va",
    "scnt_hema median velocity_cosine: labelling–dynamo_gene, labelling–scvelo, scvelo–dynamo_gene", "values",
    lambda: [vsc("labelling", "dynamo_gene"), vsc("labelling", "scvelo"), vsc("scvelo", "dynamo_gene")], ["−0.05", "0.03", "0.15"])
row(S, "agreement with Dynamo rises from ≈ 0", "S11", "va", "scnt_hema labelling vs dynamo_gene median velocity_cosine (shared-γ convention)", "approx0",
    lambda: vsc("labelling", "dynamo_gene"))
nr(S, "to 0.41 (exploratory diagnostic", "S11 (review)", "value appears only in a review note (coord/orient/reviews/labelling_diag_2026-10-19), no committed result table")

## Results 6 — score operator

In [9]:
S = "Results 6"
row(S, "on four non-stationary presets", "S10", "ops", "presets with J_score rows (PCA, saddle)", "count", lambda: OPS[OPS.operator == "J_score"].preset.nunique(), 4)
row(S, "(AUC 0.16–0.31 on default", "S10", "ops", "PCA, v3_default, J_score", "range", lambda: opauc("J_score"), "0.16–0.31")
row(S, "0.15–0.50 on stress", "S10", "ops", "PCA, v3_stress, J_score", "range", lambda: opauc("J_score", "v3_stress"), "0.15–0.50")
row(S, "0.17–0.29 on batch", "S10", "ops", "PCA, v3_batch, J_score", "range", lambda: opauc("J_score", "v3_batch"), "0.17–0.29")
sj = np.r_[opauc("J_score"), opauc("J_score", "v3_stress"), opauc("J_score", "v3_batch")]
row(S, "with AUC < 0.45 on 8 of these 9 seeds", "S10", "ops", "J_score on default/stress/batch: rows < 0.45; rows", "values",
    lambda: [int((sj < 0.45).sum()), len(sj)], ["8", "9"])
row(S, "and 0.43–0.52 on transition", "S10", "ops", "PCA, v3_transition, J_score", "range", lambda: opauc("J_score", "v3_transition"), "0.43–0.52")
row(S, "On dyngen it reached 0.98, 0.58 and 0.85 on the three bifurcations", "S21b", "dsc", "operator score, bifurcating s10/s16/s18", "values",
    lambda: [scj["bifurcating_v2_s10_T8"], scj["bifurcating_v2_s16_T8"], scj["bifurcating_v2_s18_T8"]], ["0.98", "0.58", "0.85"])
c20 = cap[(cap.k == 20) & (cap.m == 3)]
row(S, "(10,691 non-targeting controls; 865 of 1,299 targets with ≥ 100 cells passed a BH-adjusted permutation test)", "S18", ["pemb", "eff"],
    "embedding.json n_controls; effect_tests.json n_targets_significant, n_targets_eligible, minimum_cells", "values",
    lambda: [pemb["n_controls"], eff["n_targets_significant"], eff["n_targets_eligible"], eff["minimum_cells"]], ["10691", "865", "1299", "100"])
row(S, "three soft directions of the local score operator captured a median 4.5 % of each knockdown's centroid shift, against 58.9 % for the top three directions of the global control covariance and a random-subspace expectation of 15 %",
    "S18", "cap", "k 20, m 3 (865 targets): m; median local_score_J ×100; median global_covariance ×100; random_expectation ×100", "values",
    lambda: [c20.m.iloc[0], c20.local_score_J.median() * 100, c20.global_covariance.median() * 100, c20.random_expectation.iloc[0] * 100], ["3", "4.5", "58.9", "15"])
row(S, "The pre-registered criterion (≥ 70 % of targets above their random-null 95th percentile) reached 5.3 %", "S18", ["cap", "s18md"],
    "mean(local_score_exceeds_null95) ×100 (k 20, m 3); required fraction from RESULT_S18.md", "values",
    lambda: [float(re.search(r"required >=([0-9]+\.[0-9]+)", s18md).group(1)) * 100, c20.local_score_exceeds_null95.mean() * 100], ["70", "5.3"])
hh = h1[h1.k == 20]
row(S, "Re-weighting control states explained a median 31 % of each knockdown's energy distance (required ≥ 50 %)", "S18b", ["h1", "s18bmd"],
    "h1.csv k 20: median explained_fraction ×100; required_median from RESULT_S18b.md ×100", "values",
    lambda: [hh.explained_fraction.median() * 100, float(re.search(r'"required_median": ([0-9.]+)', s18bmd).group(1)) * 100], ["31", "50"])
j20 = h2["by_k"]["20"]
row(S, "across 59 basins (partial Spearman ρ +0.05, one-sided p 0.65; Fig. 6c)", "S18b", "h2", "by_k 20: n_basins; stiffness_partial rho, p_one_sided_negative", "values",
    lambda: [j20["n_basins"], j20["stiffness_partial"]["rho"], j20["stiffness_partial"]["p_one_sided_negative"]], ["59", "+0.05", "0.65"])
row(S, "underpowered (1 qualifying target; ≥ 10 required)", "S18d", "dg", "gates.json outcome == underpowered (1 = yes); n_T rpe1; underpowered_threshold_n_T", "values",
    lambda: [int(dg["outcome"] == "underpowered" and dg["underpowered"]), dg["n_T"]["rpe1"], dg["underpowered_threshold_n_T"]], ["1", "1", "10"], note="first value: outcome flag")
row(S, "In K562 (report only), score-operator capture of weak-knockdown displacements did not exceed a shared-basis null or the covariance baseline", "S18d", "dg",
    "k562 weak, N1 and N2: neither D1 (median > null95) nor D2 (Wilcoxon S_J > S_C p < 0.05) met; K562 not primary", "bool",
    lambda: dg["primary_screen"] != "k562" and not any(dg["comparisons"]["k562"][n_]["weak"][d_] for n_ in ["N1", "N2"] for d_ in ["D1_criterion_met", "D2_criterion_met"]), True)
g22 = s22[s22.seed == 0].set_index("cohort")
row(S, "The pooled correlation between clone λ₁ and fate entropy was ρ = −0.003", "S22", "s22", "seed 0, cohort pooled: rho_lambda1_H", "value",
    lambda: g22.loc["pooled", "rho_lambda1_H"], "−0.003")
row(S, "within undifferentiated cells it was −0.10", "S22", "s22", "seed 0, cohort undifferentiated: rho_lambda1_H", "value",
    lambda: g22.loc["undifferentiated", "rho_lambda1_H"], "−0.10")
row(S, "(|ρ| 0.22 vs 0.10 for λ₁)", "S22", "s22", "seed 0, undifferentiated: |rho_centroid_distance_H|, |rho_lambda1_H|", "values",
    lambda: [abs(g22.loc["undifferentiated", "rho_centroid_distance_H"]), abs(g22.loc["undifferentiated", "rho_lambda1_H"])], ["0.22", "0.10"])
row(S, "the partial correlation given local density was −0.08", "S22", "s22", "seed 0, undifferentiated (L3 cohort): partial_rho_given_density", "value",
    lambda: g22.loc["undifferentiated", "partial_rho_given_density"], "−0.08")
row(S, "All three gates failed", "S22", "s22sum", "summary.json gate_L1/L2/L3 passed == False (count)", "count",
    lambda: sum(not s22sum[k_]["passed"] for k_ in ["gate_L1_pooled", "gate_L2_undifferentiated", "gate_L3_baselines"]), 3)


## Results 7 — archetypes (S15)

In [10]:
S = "Results 7"
oa1 = {d["dataset"]: d for d in A1["operator_archetype"]["saddle"]}; dk1 = {d["dataset"]: d for d in A1["descriptor_kmeans"]["saddle"]}
c2 = {c["dataset"]: c for c in A2["operator_archetype"]["comparisons"]}
c2d = {c["dataset"]: c for c in A2["descriptor_kmeans"]["comparisons"]}
row(S, "(operator-archetype view, K = 8)", "S15", "s15g", "gate_A2 operator_archetype: k_selected over test seeds", "values",
    lambda: sorted(set(c["k_selected"] for c in c2.values())), ["8"])
row(S, "(descriptor view, K = 2)", "S15", "s15g", "gate_A2 descriptor_kmeans: k_selected over test seeds", "values",
    lambda: sorted(set(c["k_selected"] for c in c2d.values())), ["2"])
row(S, "on the three held-out transition seeds", "S15", "s15g", "datasets in gate_A1 operator_archetype block", "count", lambda: len(oa1), 3)
row(S, "recall 0.72–0.79 at 3.4–4.1× their base rate", "S15", "s15g", "gate_A1 operator_archetype: recall min/max, precision_over_base_rate min/max", "values",
    lambda: [min(d["recall"] for d in oa1.values()), max(d["recall"] for d in oa1.values()),
             min(d["precision_over_base_rate"] for d in oa1.values()), max(d["precision_over_base_rate"] for d in oa1.values())], ["0.72", "0.79", "3.4", "4.1"])
row(S, "The descriptor view had higher recall (0.88–0.95)", "S15", "s15g", "gate_A1 descriptor_kmeans: recall", "range", lambda: [d["recall"] for d in dk1.values()], "0.88–0.95")
row(S, "but failed precision on one seed (1.5×)", "S15", "s15g", "gate_A1 descriptor_kmeans: seeds below 3× and their min precision_over_base_rate", "values",
    lambda: [sum(d["precision_over_base_rate"] < 3 for d in dk1.values()), min(d["precision_over_base_rate"] for d in dk1.values())], ["1", "1.5"])
row(S, "normalized mutual information was 0.35–0.39 for flow archetypes", "S15", "s15g", "gate_A2 operator_archetype: nmi_flow", "range", lambda: [c["nmi_flow"] for c in c2.values()], "0.35–0.39")
row(S, "0.35–0.37 for Leiden at matched K", "S15", "s15g", "gate_A2 operator_archetype: leiden_nmi", "range", lambda: [c["leiden_nmi"] for c in c2.values()], "0.35–0.37")
row(S, "0.05–0.31 for score archetypes", "S15", "s15g", "gate_A2 operator_archetype: nmi_score", "range", lambda: [c["nmi_score"] for c in c2.values()], "0.05–0.31")
row(S, "Flow beat score on every seed but Leiden on only one (gate A2 fails)", "S15", "s15g", "seeds nmi_flow > nmi_score; seeds nmi_flow > leiden_nmi (operator view)", "values",
    lambda: [sum(c["nmi_flow"] > c["nmi_score"] for c in c2.values()), sum(c["nmi_flow"] > c["leiden_nmi"] for c in c2.values())], ["3", "1"])
a3o = [d["ari"] for d in A3["operator_archetype"]["drift_seed_ari"]]; a3d = [d["ari"] for d in A3["descriptor_kmeans"]["drift_seed_ari"]]
row(S, "between seeds was 0.22–0.44 in the operator view", "S15", "s15g", "gate_A3 operator_archetype drift_seed_ari (9 pairs)", "range", lambda: a3o, "0.22–0.44")
row(S, "and −0.02 to 0.53 in the descriptor view", "S15", "s15g", "gate_A3 descriptor_kmeans drift_seed_ari (9 pairs)", "range", lambda: a3d, "−0.02 to 0.53")
row(S, "(gate A3, ≥ 0.5 for every pair, fails)", "S15", "s15g", "any view with all pairs >= 0.5 (recomputed) ; gate A3 pass flag", "values",
    lambda: [int(all(a >= 0.5 for a in a3o) or all(a >= 0.5 for a in a3d)), int(s15g["gates"]["gate_A3_stability"]["pass"])], ["0", "0"])

## Discussion

In [11]:
S = "Discussion"
row(S, "on an independent simulator it did so on two of three bifurcations", "S21b", "dsc", "flow; bifurcations with fork_auc >= 0.85 on all drift seeds", "count",
    lambda: int((bif.groupby("dataset").fork_auc.min() >= 0.85).sum()), 2)
row(S, "it held in PCA of log-normalized counts with 10–20 components, degraded at 50 components", "S12", "s12",
    "min AUC: pca10/pca20 (>= 0.80) vs pca50; embedding dims", "values",
    lambda: [10, 20, 50] if (min(auc(s12[s12.embedding.isin(["pca10", "pca20"])])) > min(auc(s12[s12.embedding == "pca50"]))) else [np.nan] * 3, ["10", "20", "50"],
    note="checked: min fork AUC PCA-10/20 > min fork AUC PCA-50")
row(S, "Recovery of Jacobian values, for either operator (relative error ≈ 1.0)", "S14/S15", ["test", "jref"], "flow: S14 jref_relative_frobenius_median and S15 cstar error (score operator: no committed J_ref table)", "approx1",
    lambda: np.r_[syn.jref_relative_frobenius_median.dropna().values, jref.median_relative_error_cstar_J.values], note="score-operator half of the claim not recomputable here")
sm_ = lambda meth: [dsum["units"][f"v3_default_s{s_}_{meth}"]["sigma_meas_median_trace_over_d"] for s_ in range(3)]
tr48 = lambda meth: [diff[(diff.preset == "v3_default") & (diff.seed == s_) & (diff.method == meth) & (diff.estimator == "clone_dispersion") & (diff.variant == "uncorrected")].median_truth_all.iloc[0] for s_ in range(3)]
row(S, "count noise in the embedding exceeds 48-h diffusion (Supplementary Fig. S1)", "S10", ["dsum", "diff"], "Σ_meas trace/d > true 48-h trace(D_eff)/d for every seed, PCA and FA", "bool",
    lambda: all(a > b for m_ in ["pca", "fa"] for a, b in zip(sm_(m_), tr48(m_))), True)
row(S, "Fork detection on every independent-simulator bifurcation (2 of 3) and rotation on every cycle (2 of 3)", "S21b", "dsc", "bifurcations passing; cycles passing", "values",
    lambda: [int((bif.groupby("dataset").fork_auc.min() >= 0.85).sum()), n_cycles()], ["2", "2"])
nr(S, "(search date 2026-10-26)", "—", "date of the literature search (manuscript statement; manuscript/LITERATURE_REVIEW_v57.md is not a result file)")
row(S, "met only 6 of 13 pre-fixed calibration targets per seed", "S09", "gates3", "gates_v3.json realism: met_count_by_seed (min, max), n_statistics", "values",
    lambda: [min(rz["met_count_by_seed"].values()), max(rz["met_count_by_seed"].values()), rz["n_statistics"]], ["6", "6", "13"])

## Methods

In [12]:
S = "Methods: Embedding for flow operators"
nr(S, "2,000 highly variable genes were selected (seurat_v3 flavor on counts)", "S14", CFG)
row(S, "PCA with k = 15 components (k = 10–20 tested)", "S14", ["ksel", "test"], "k_star; k values in test.csv", "values",
    lambda: [ksel.get("k_star", selcfg["k_star"]), min(test.k), max(test.k)], ["15", "10", "20"])
row(S, "LARRY real-data comparisons in Supplementary Fig. S4 used PCA-30", "S11", "s11md", "RESULT_S11.md: 'fixed common PCA-30 embeddings'", "bool",
    lambda: "PCA-30" in s11md, True)
S = "Methods: Time-course velocity"
nr(S, "ε = 0.05 × median squared distance", "S10", CFG)
nr(S, "the mini-batch variant uses 4,000-cell source batches in which every cell appears in exactly ten batches", "S20", CFG)
row(S, "Low-rank OT used moscot at rank 50 and 200 [27]", "S20", "agr", "low-rank solver ranks present", "values",
    lambda: sorted(int(s_.split("_r")[1]) for s_ in lr.solver.unique()), ["50", "200"])
S = "Methods: Drift and flow operator"
fz = s15md.split("## Frozen configuration", 1)[1].split("##", 1)[0]
row(S, "three hidden layers of 96 units, weight decay 1e-4, single member, seed 0 by default", "S14/S15", "s15md",
    "RESULT_S15.md 'Frozen configuration' contains '96x96x96 drift', 'weight decay 1e-4', 'one member'", "bool",
    lambda: all(s_ in fz for s_ in ["96x96x96 drift", "weight decay 1e-4", "one member"]), True, note="seed 0 default: S11 measured_s0 rows")
row(S, "seeds 1–2 for variability", "S11", "oa", "S11 seed-variability estimators present (measured_s1, measured_s2)", "bool",
    lambda: {"measured_s1", "measured_s2"} <= set(oa.right_estimator), True)
cf = selcfg["configuration"]
row(S, "The tuned configuration of the S14 test was whitened PCA-15, ε factor 0.025, extrapolation, MLP 128×128×128, weight decay 1e-3 and a five-member ensemble",
    "S14", "selcfg", "selected_configuration.json: k_star, eps, hidden, wd, ensemble (space whitened, last extrapolate)", "values",
    lambda: [selcfg["k_star"], cf["eps"], cf["hidden"][0] if (cf["space"] == "whitened" and cf["last"] == "extrapolate" and len(set(cf["hidden"])) == 1 and len(cf["hidden"]) == 3) else np.nan, cf["wd"] * 1000, cf["ensemble"]],
    ["15", "0.025", "128", "1", "5"], note="weight decay compared as wd×1000 = 1")
S = "Methods: Score operator"
nr(S, "An energy network (96 hidden units, depth 3)", "S10", CFG)
nr(S, "(1,500 epochs, learning rate 5e-4, batch size 256)", "S10", CFG)
nr(S, "(β = 1 in the S11/S22 fits)", "S11/S22", CFG)
nr(S, "(maximum asymmetry ≤ 1e-4)", "S10", "threshold; observed value only in RESULT_S10.md prose, no per-fit table read here")
S = "Methods: Descriptors and scoring"
nbin = test.transition_bin_medians.dropna().map(lambda s_: len((lambda x: _json.loads(x) if isinstance(x, str) else x)(_json.loads(s_))))
row(S, "the per-bin median λ₁ over 12 day-weighted strict-pseudotime bins", "S14", "test", "length of transition_bin_medians (all rows): min, max", "values",
    lambda: [int(nbin.min()), int(nbin.max())], ["12", "12"])
row(S, "(kNN neighborhood 15 or 30)", "S15", "jref", "jref_neighbours values", "values", lambda: sorted(jref.jref_neighbours.unique()), ["15", "30"])
S = "Methods: Calibrated synthetic generator"
row(S, "Twenty real-data statistics (S08) were computed on six datasets", "S08", ["cmp8", "stats"], "distinct statistic IDs; real datasets", "values",
    lambda: [len(ids(cmp8)), stats[stats.kind == "real"].dataset.nunique()], ["20", "6"])
row(S, "calibrated against 13 targets fixed in advance", "S09", "gates3", "gates_v3.json realism n_statistics", "count", lambda: rz["n_statistics"], 13)
row(S, "sampled at days 2, 4 and 6", "S10", "byday", "distinct day values in by_day.csv", "values", lambda: sorted(byday.day.unique()), ["2", "4", "6"])
S = "Methods: Held-out test of the flow operator (S14)"
row(S, "Part A chose k from {5, 10, 15, 20, 30}", "S14", "choosek", "k values in choose_k.csv", "values", lambda: sorted(choosek.k.unique()), ["5", "10", "15", "20", "30"])
row(S, "(operator seed stability ≥ 0.5 required before lineage agreement counts)", "S14", "ksel", "k_selection_r1.json stability_threshold", "value",
    lambda: ksel["stability_threshold"], "0.5")
row(S, "giving k* = 15", "S14", "ksel", "k_selection_r1.json k_star (selected_configuration.json k_star)", "values",
    lambda: [ksel.get("k_star", selcfg["k_star"]), selcfg["k_star"]], ["15", "15"])
row(S, "on two development seeds", "S14", "dev", "development seeds per preset", "count", lambda: int(dev.groupby("preset").seed.nunique().max()), 2)
dsn = lambda pr: sorted(int(s_.rsplit("_s", 1)[1]) for s_ in syn[syn.preset == pr].dataset.unique())
row(S, "(v3_transition s2–s4, v3_default s2–s3, v3_batch s0–s2, v3_stress s0–s2 and v3_nofork s0–s2) at k = 10, 15 and 20", "S14", "test",
    "test.csv seeds per preset (transition, default, batch, stress, nofork: min,max each) and k values", "values",
    lambda: sum([[min(dsn(p)), max(dsn(p))] for p in ["v3_transition", "v3_default", "v3_batch", "v3_stress", "v3_nofork"]], []) + sorted(syn.k.unique()),
    ["2", "4", "2", "3", "0", "2", "0", "2", "0", "2", "10", "15", "20"])
row(S, "and on LARRY at k = 15", "S14", "test", "kind larry: k values", "values", lambda: sorted(test[test.kind == "larry"].k.unique()), ["15"])
row(S, "Gates G1–G4 applied to the tuned configuration", "S14", "gates14", "number of gate_G* entries in gates.json", "count", lambda: sum(k.startswith("gate_G") for k in gates14), 4)
S = "Methods: Transition eligibility (S14b, S14c)"
row(S, "(b) every bin had ≥ 50 cells", "S14b/S14c", "elig", "threshold 50 reproduces all_bins_at_least_50_cells on every assessed seed (1 = yes)", "count",
    lambda: int(all((el[done].min_bin >= 50) == T(el[done].all_bins_at_least_50_cells))), 1)
row(S, "(c) the true peak bin was in bins 2–9", "S14b/S14c", "elig", "rule 2 <= true_peak_bin <= 9 reproduces true_peak_bin_2_to_9 on every assessed seed (1 = yes)", "count",
    lambda: int(all(el[done].true_peak_bin.between(2, 9) == T(el[done].true_peak_bin_2_to_9))), 1)
row(S, "(d) the argmax matched the true bin in ≥ 90 % of 200 bootstrap resamples within bins", "S14b/S14c", "elig",
    "threshold 0.90 reproduces bootstrap flag (1 = yes); bootstrap_replicates", "values",
    lambda: [int(all((el[done].bootstrap_stability >= 0.9) == T(el[done].bootstrap_stability_at_least_90pct))), el[done].bootstrap_replicates.max()], ["1", "200"])
row(S, "Four eligible transitions were required", "S14c", "g14c", "gates.json required_eligible_seeds", "count", lambda: int(g14c["required_eligible_seeds"]), 4)
row(S, "S14b used reserve seeds 5–16, and S14c seeds 17–40 with a 12 GB generation memory guard", "S14b/S14c", ["eligb", "elig", "md14c"],
    "S14b seeds min/max; S14c seeds min/max; guard threshold bytes/1e9 from RESULT_S14c.md", "values",
    lambda: [eligb.seed.min(), eligb.seed.max(), c14.seed.min(), c14.seed.max(), int(re.search(r">= (\d+)\)", md14c).group(1)) / 1e9], ["5", "16", "17", "40", "12"])
S = "Methods: Independent simulator"
nr(S, "dyngen 1.1.1", "S19", "software version; dyngen settings files are not committed")
row(S, "with backbones bifurcating, cycle and linear (three datasets each), 5,000 cells, 1,000 genes", "S21b", "dman", "MANIFEST _T8 rows: datasets per backbone; cells; genes", "values",
    lambda: [t8.groupby("backbone").size().min(), t8.groupby("backbone").size().max(), t8.cells.max(), t8.genes.max()], ["3", "3", "5000", "1000"])
nr(S, "64 simulations for bifurcating backbones", "S19", "dyngen setting; settings files not committed")
row(S, "synchronized experiments with 8 (S21b) or 3 (S21) time points", "S21b", "dpa", "part_a_T.csv T", "values", lambda: sorted(dpa["T"].unique()), ["8"],
    note="the S21 value (3) is not recorded in a committed S21 result table")
S = "Methods: LARRY"
row(S, "130,887 cells", "S20", "full", "sum n_cells", "count", lambda: int(full.n_cells.sum()), 130887)
row(S, "fixed sample of 4,000 cells per time point", "S14", "test", "kind larry n_cells / 3", "count", lambda: int(larry.loc["reference", "n_cells"] / 3), 4000)
row(S, "The pre-registered test used five permutations; post hoc we used 1,000", "S14", ["test", "add14"], "length of lineage_nulls (reference); addendum n_permutations", "values",
    lambda: [len(_json.loads(larry.loc["reference", "lineage_nulls"])), add14["n_permutations"]], ["5", "1000"])
nr(S, "(1-h Euler steps)", "S14", CFG)
row(S, "resample day-4 and day-6 cells (200 resamples)", "S14 addendum", "add14", "heldout_energy n_boot", "values", lambda: [4, 6, add14["heldout_energy"]["n_boot"]], ["4", "6", "200"])
nr(S, "fits transport and drift on days 2 and 4 only, integrates the drift from day-4 cells over the 48-h gap (1-h Euler steps) and compares the result with observed day-6 cells by energy distance in a common PCA-50 space", "S14", CFG)
row(S, "the gene selection and PCA were fitted on the 12,000-cell sample of all three days", "S14", "test", "larry n_cells in S14 checkpoint (3 x 4,000)", "values", lambda: [12000, 3], ["12,000", "3"], note="prepare_larry: S13 common map on the fixed default_rng(0) sample")
nr(S, "day-(*t* + 2) cells", "S14", "notation: next sampling day (2-day spacing)")
nr(S, "so day 6 contributes to the coordinate system", "S14", "sampling day label")
S = "Methods: Scaling (S20)"
row(S, "Dense, mini-batch and low-rank OT were run at 2,000–16,000 cells per day", "S20", "scal", "numeric n_per_day values in scaling.csv", "range",
    lambda: [int(v) for v in scal.n_per_day if v != "full"], "2,000–16,000")
S = "Methods: Archetypes (S15)"
row(S, "Per-cell operators of the reference flow operator (k = 15)", "S15", "rr", "regime_recovery.csv k of the gate rows (flow)", "values",
    lambda: sorted(set(d["k"] for d in A1["operator_archetype"]["saddle"])), ["15"])
nr(S, "with K chosen by held-out reconstruction over {2, 3, 4, 5, 6, 8}", "S15", CFG + "; selected K values in archetypes.csv are " + ", ".join(map(str, sorted(arch.k_selected.unique()))))
nr(S, "K by mean silhouette over five restarts", "S15", CFG)
nr(S, "Leiden clustering on expression PCA-15", "S15", CFG)
row(S, "ARI across drift seeds 0–2", "S15", "stab", "drift seeds in stability.csv drift_seed pairs", "values",
    lambda: sorted(set(stab[stab.comparison == "drift_seed"].left.astype(int)) | set(stab[stab.comparison == "drift_seed"].right.astype(int))), ["0", "1", "2"])
row(S, "A1 (an archetype captures saddle cells with recall ≥ 0.6 and precision ≥ 3× base rate on every test seed)", "S15", "s15g",
    "recomputed pass with these thresholds equals the gate flag, both views (1 = yes)", "count",
    lambda: int(all(all(d["recall"] >= 0.6 and d["precision_over_base_rate"] >= 3 for d in A1[v]["saddle"]) == A1[v]["pass"] for v in ["operator_archetype", "descriptor_kmeans"])), 1)
row(S, "A3 (ARI ≥ 0.5 across drift seeds)", "S15", "s15g", "recomputed pass with ARI >= 0.5 equals the gate flag, both views (1 = yes)", "count",
    lambda: int(all(all(d["ari"] >= 0.5 for d in A3[v]["drift_seed_ari"]) == A3[v]["pass"] for v in ["operator_archetype", "descriptor_kmeans"])), 1)
S = "Methods: Perturbation tests (S18, S18b, S18d)"
row(S, "(median library size, log1p, 2,000 control seurat_v3 HVGs, control-only PCA-20)", "S18", "pemb", "embedding.json: n_hvg, hvg_flavor seurat_v3, 20 in pca_k, pca fit on controls", "values",
    lambda: [pemb["n_hvg"], 20 if (pemb["hvg_flavor"] == "seurat_v3" and 20 in pemb["pca_k"] and "control" in pemb["pca_fit_cells"]) else np.nan], ["2000", "20"])
row(S, "Targets with ≥ 100 cells were tested", "S18", "eff", "effect_tests.json minimum_cells", "count", lambda: eff["minimum_cells"], 100)
row(S, "captured by three directions", "S18", "cap", "capture.csv m of the primary rows", "values", lambda: sorted(c20.m.unique()), ["3"])
row(S, "re-weighting control states (k = 20)", "S18b", "h2", "h2.json primary_k", "count", lambda: h2["primary_k"], 20)
row(S, "(10,000 permutations)", "S18b", "h2", "h2.json by_k 20 stiffness_partial permutations", "count", lambda: j20["stiffness_partial"]["permutations"], 10000)
def _weak_lowest():
    ok = True
    for (s_, n_), d_ in dcap.items():
        p_ = d_.pivot_table(index="target", columns="tertile", values="median_knockdown")
        ok &= bool(((p_.weak <= p_.medium) & (p_.medium <= p_.strong)).all())
    return ok
row(S, "(lowest knockdown tertile)", "S18d", ["dcap_rpe1_N1", "dcap_rpe1_N2", "dcap_k562_N1", "dcap_k562_N2"],
    "capture_*.csv: weak tertile has the lowest median_knockdown of every target (weak ≤ medium ≤ strong)", "bool", _weak_lowest, True)
S = "Methods: Clonal fate coherence (S22)"
row(S, "LARRY PCA-30 (4,000 cells per day; seeds 0–2, gates on seed 0)", "S22", ["s22cells", "s22fit", "s22pre"],
    "eigenvalue dimension (cells_seed0.npz); n_training_cells / 3; score_config seeds min/max; gated_seed", "values",
    lambda: [S22_DIM, s22fit["n_training_cells"] / 3, min(s22pre["score_config"]["seeds"]), max(s22pre["score_config"]["seeds"]), s22pre["gated_seed"]], ["30", "4,000", "0", "2", "0"])
row(S, "every barcoded day-2 and day-4 cell", "S22", ["s22fit", "s22md"], "fit n_query_cells equals the 'Every one of the N barcoded day-2/day-4 cells' count in RESULT_S22.md", "bool",
    lambda: s22fit["n_query_cells"] == int(re.search(r"Every one of the ([0-9,]+) barcoded day-2/day-4 cells", s22md).group(1).replace(",", "")), True)
row(S, "For clones with ≥ 2 cells on day *t* and ≥ 3 cells on day *t* + 2", "S22", "s22cl", "clones.csv min n_current, min n_future", "values",
    lambda: [s22cl.n_current.min(), s22cl.n_future.min()], ["2", "3"], note="minimum observed equals the stated threshold")
row(S, "the day-(*t* + 2) cell-type distribution", "S22", "s22cl", "clones.csv distinct t; future day = t + 2", "values",
    lambda: sorted(s22cl.t.unique()), ["2", "4"], note="t ∈ {2, 4}; future days 4 and 6")
row(S, "distance to the day-2 undifferentiated centroid", "S22", "s22md", "RESULT_S22.md states 'all-day-2 Undifferentiated centroid'", "bool",
    lambda: "all-day-2 Undifferentiated centroid" in s22md, True)
row(S, "with 1,000 within-stratum permutations", "S22", ["s22pre", "s22null"], "preregistration permutations; null array lengths (seed 0)", "values",
    lambda: [s22pre["permutations"], S22_NPERM.pop() if len(S22_NPERM) == 1 else np.nan], ["1,000", "1,000"])
S = "Methods: Diffusion (S10)"
row(S, "estimated by clone dispersion and OT noise (Δt = 48 h) and by a score–flow estimator (6 h)", "S10", "dsum", "gate_diffusion definition mentions 'D_eff(48 h)' and 'D_eff(6 h)'", "bool",
    lambda: "D_eff(48 h)" in dsum["gate_diffusion"]["definition"] and "D_eff(6 h)" in dsum["gate_diffusion"]["definition"], True)
nr(S, "the truth from 32 forward paths per cell", "S10", "simulation setting; not recorded in diffusion.csv / diffusion_summary.json")
S = "Methods: Statistics"
row(S, "five permutations for the pre-registered LARRY lineage test (1,000 post hoc) and for S11 operator agreement; with five permutations the smallest attainable p is 1/6", "S14/S11", ["test", "add14", "nu"], "len(lineage_nulls); addendum n_permutations; null.csv permutations per larry independent_sources block; 1/(5+1)", "values",
    lambda: [len(_json.loads(larry.loc["reference", "lineage_nulls"])), add14["n_permutations"], len(nl), 1, 6], ["5", "1000", "5", "1", "6"])
row(S, "1,000 within-stratum permutations (S22)", "S22", "s22pre", "preregistration permutations", "count", lambda: s22pre["permutations"], 1000)
row(S, "10,000 (S18b)", "S18b", "h2", "h2.json permutations", "count", lambda: j20["stiffness_partial"]["permutations"], 10000)
row(S, "bootstrap stability used 200 resamples", "S14b/S14c", "elig", "bootstrap_replicates (assessed seeds)", "count", lambda: int(el[done].bootstrap_replicates.max()), 200)
S = "Methods: Software"
vers = ["Python 3.13.2", "NumPy 2.5.4", "pandas 2.3.3", "AnnData 0.11.4", "Scanpy 1.12.4", "Palantir 1.4.5", "SciPy 1.18.1", "scikit-learn 1.9.1", "scvelo 0.3.4", "PyTorch 2.14.1"]
def _conflicts(md):
    n = 0
    for v_ in vers:
        name, ver = v_.rsplit(" ", 1)
        m_ = re.search(re.escape(name) + r" (\d[\w.]*)", md)
        n += bool(m_ and m_.group(1).rstrip(".") != ver)
    return n
row(S, "Python 3.13.2, NumPy 2.5.4, pandas 2.3.3, AnnData 0.11.4, Scanpy 1.12.4, Palantir 1.4.5, SciPy 1.18.1, scikit-learn 1.9.1, scvelo 0.3.4, PyTorch 2.14.1",
    "S14b/S14c", ["md14c", "md14b"], "version strings found verbatim in RESULT_S14c.md (of 10); packages whose version in RESULT_S14b.md differs", "values",
    lambda: [sum(v_ in md14c for v_ in vers), _conflicts(md14b)], ["10", "0"],
    note="RESULT_S14b.md records " + ", ".join(v_ for v_ in vers if v_ in md14b) + "; it does not record " + ", ".join(v_.rsplit(" ", 1)[0] for v_ in vers if v_ not in md14b))

## Figure legends

In [13]:
S = "Legend Fig 2"
row(S, "Six of the 20 S08 statistics (IDs C1–C8, G1–G5, L1–L2, S1–S4, T2; 32 sub-statistics)", "S08", "cmp8",
    "distinct IDs (and their ranges per letter); distinct sub-statistics", "values",
    lambda: [len(ids(cmp8))] + [int(max(int(i[1:]) for i in ids(cmp8) if i[0] == L_)) for L_ in "CGLS"] + [cmp3.statistic.nunique()],
    ["20", "8", "5", "2", "4", "32"], note="IDs present: " + ", ".join(ids(cmp8)) + "; sub-statistics counted in compare_v3.csv (the S08 comparison table as extended by S09, plotted in Fig. 2); S08 compare.csv has " + str(cmp8.statistic.nunique()) + ", stats.csv has " + str(stats.statistic.nunique()) + " statistic labels")
zz = cmp3[cmp3.statistic == "C5_zero_fraction"]; gen = lambda p: "v1" if p == "v1" else p.split("_")[0]
zmed = lambda gname: float(np.nanmedian(zz[zz.preset.map(gen) == gname].synthetic_scalar_or_median.astype(float)))
row(S, "Zero fraction: real 0.84–0.96 vs v1 median 0.36, v2 0.37, v3 0.87", "S09", "cmp3",
    "C5_zero_fraction: real min/max of real_values_or_medians; median synthetic value per generator (v1, v2 presets, v3 presets)", "values",
    lambda: [zz.real_range_min.iloc[0], zz.real_range_max.iloc[0], zmed("v1"), zmed("v2"), zmed("v3")], ["0.84", "0.96", "0.36", "0.37", "0.87"])
row(S, "(v1: 3 seeds of the default preset; v2 and v3: 3 seeds × each preset)", "S09", "cmp3", "max seeds per preset (v1, v2_*, v3_*): synthetic datasets per preset", "values",
    lambda: [cmp3[cmp3.preset == "v1"].synthetic.nunique(), cmp3[cmp3.preset.str.startswith("v2")].groupby("preset").synthetic.nunique().max(), cmp3[cmp3.preset.str.startswith("v3")].groupby("preset").synthetic.nunique().max()], ["3", "3", "3"])
row(S, "v1 0.35, v2_default 0.22, v3_default 0.39, v3_batch 0.39, v3_stress 0.34, v3_stationary 0.49", "S09", "cmp3", "mean(match) per preset", "values",
    lambda: [mr[p] for p in ["v1", "v2_default", "v3_default", "v3_batch", "v3_stress", "v3_stationary"]], ["0.35", "0.22", "0.39", "0.39", "0.34", "0.49"])
row(S, "dyngen (S19, nine datasets, bifurcating/cycle/linear × 3 seeds, 25 applicable statistics) 0.51 (range 0.44–0.60)", "S19", "dynr",
    "datasets; statistics per dataset; mean match; per-dataset min–max", "values",
    lambda: [dynr.synthetic.nunique(), dynr.groupby("synthetic").size().max(), dmt.m.mean(), dmt.groupby("synthetic").m.mean().min(), dmt.groupby("synthetic").m.mean().max()],
    ["9", "25", "0.51", "0.44", "0.60"])
row(S, "v3_default met 6 of 13 on each seed (realism gate failed)", "S09", "gates3", "met_count_by_seed min/max; n_statistics; pass", "values",
    lambda: [min(rz["met_count_by_seed"].values()), max(rz["met_count_by_seed"].values()), rz["n_statistics"], int(rz["pass"])], ["6", "6", "13", "0"])
row(S, "targets unmet on all three seeds: C2_genes_detected, G3_multi_time_clone_fraction, S1_unspliced_fraction, S3_steady_share, S4_recovered_fraction", "S09", "gates3",
    "intersection of unmet_by_seed", "bool", lambda: ", ".join(sorted(set.intersection(*[set(v) for v in rz["unmet_by_seed"].values()]))),
    "C2_genes_detected, G3_multi_time_clone_fraction, S1_unspliced_fraction, S3_steady_share, S4_recovered_fraction")
row(S, "Empirical CDFs of three saved S08 distributions for five real datasets and v1/v2 default seed 0", "S08", None,
    "real datasets with saved C3/C4/C5 distributions in benchmarks/real_reference/distributions", "count",
    lambda: len({p.name.split("__")[0] for p in resolve("benchmarks/real_reference/distributions").glob("*__C5_zero_fraction.csv") if not p.name.startswith(("v1_", "v2_"))}), 5)
row(S, "Against the 13 pre-fixed S09 calibration targets", "S09", "gates3", "n_statistics", "count", lambda: rz["n_statistics"], 13)
row(S, "the S09 13-target count", "S09", "gates3", "n_statistics", "count", lambda: rz["n_statistics"], 13)

S = "Legend Fig 3"
row(S, "(days 2/4/6)", "S10", "byday", "distinct day values", "values", lambda: sorted(byday.day.unique()), ["2", "4", "6"])
a_sets = ref[ref.preset.isin(["v3_transition", "v3_default", "v3_batch", "v3_stress"])]
row(S, "11 test datasets (v3_transition s2–s4, v3_default s2–s3, v3_batch s0–s2, v3_stress s0–s2) at k = 10, 15, 20", "S14", "test",
    "distinct fork-preset test datasets; k values", "values", lambda: [a_sets.dataset.nunique()] + sorted(a_sets.k.unique()), ["11", "10", "15", "20"])
row(S, "Reference (a-priori) configuration: AUC 0.89–0.97 on v3_transition and 0.97–1.00 on the other presets", "S14", "test",
    "reference: v3_transition min/max; default/batch/stress min/max", "values",
    lambda: [auc(ref[ref.preset == "v3_transition"]).min(), auc(ref[ref.preset == "v3_transition"]).max(),
             auc(ref[ref.preset.isin(["v3_default", "v3_batch", "v3_stress"])]).min(), auc(ref[ref.preset.isin(["v3_default", "v3_batch", "v3_stress"])]).max()],
    ["0.89", "0.97", "0.97", "1.00"])
row(S, "Tuned configuration (one factor at a time on two development seeds): 0.42–0.95 on v3_transition", "S14", ["dev", "test"], "dev seeds; selected v3_transition min/max", "values",
    lambda: [dev.groupby("preset").seed.nunique().max(), auc(sel[sel.preset == "v3_transition"]).min(), auc(sel[sel.preset == "v3_transition"]).max()], ["2", "0.42", "0.95"])
row(S, "(gate G1, ≥ 0.85 on every transition seed at k* = 15 and neighbors, failed; minimum 0.42)", "S14", "gates14", "gate_G1_auc_band: k*, pass, minimum_auc", "values",
    lambda: [selcfg["k_star"], int(gates14["gate_G1_auc_band"]["pass"]), gates14["gate_G1_auc_band"]["minimum_auc"]], ["15", "0", "0.42"])
row(S, "(PCA-20; 3 seeds each; filled = v3_default)", "S10", "ops", "seeds per preset × operator (PCA, saddle)", "count",
    lambda: int(OPS.groupby(["preset", "operator"]).seed.nunique().max()), 3)
row(S, "v3_default: OT flow J 0.97–1.00, labeling flow J 0.53–0.89 (raw and smoothed variants), scVelo flow J 0.27–0.51, score J 0.16–0.31", "S10", "ops",
    "v3_default min/max: J_flow_ot; raw+smoothed labelling; J_flow_scvelo; J_score", "values",
    lambda: [opauc("J_flow_ot").min(), opauc("J_flow_ot").max(), np.r_[opauc("J_flow_raw_shared"), opauc("J_flow_smoothed_shared")].min(),
             np.r_[opauc("J_flow_raw_shared"), opauc("J_flow_smoothed_shared")].max(), opauc("J_flow_scvelo").min(), opauc("J_flow_scvelo").max(),
             opauc("J_score").min(), opauc("J_score").max()], ["0.97", "1.00", "0.53", "0.89", "0.27", "0.51", "0.16", "0.31"])
e12 = lambda e: auc(s12[s12.embedding == e])
row(S, "(6 datasets: v3_default and v3_transition s0–s2)", "S12", "s12", "distinct datasets in step0.csv; seeds", "values",
    lambda: [s12.dataset.nunique() if "dataset" in s12 else s12.groupby(["preset", "seed"]).ngroups, s12.seed.min(), s12.seed.max()], ["6", "0", "2"])
row(S, "PCA-10 0.88–1.00, PCA-20 0.82–1.00, PCA-50 0.56–0.97, scVI-10 0.31–0.96, FA-20 0.04–0.53, FA-20 with noise-whitened cost 0.21–0.97", "S12", "s12",
    "min/max per embedding pca10, pca20, pca50, scvi10, fa20, fa20w", "values",
    lambda: sum([[e12(e).min(), e12(e).max()] for e in ["pca10", "pca20", "pca50", "scvi10", "fa20", "fa20w"]], []),
    ["0.88", "1.00", "0.82", "1.00", "0.56", "0.97", "0.31", "0.96", "0.04", "0.53", "0.21", "0.97"])
row(S, "whitening the embedding (PCA-20 0.37–0.58), whitening only the drift fit (PCA-10 0.79–0.96), whitening only the OT cost (PCA-10 0.53–0.79, PCA-20 0.28–0.43)", "S13", "s13",
    "v3_transition s0–2 min/max: pca20 whitened_full; pca10 raw_ot_whitened_drift; pca10 / pca20 whitened_ot_raw_drift", "values",
    lambda: [a13("pca20", "whitened_full").min(), a13("pca20", "whitened_full").max(), a13("pca10", "raw_ot_whitened_drift").min(), a13("pca10", "raw_ot_whitened_drift").max(),
             a13("pca10", "whitened_ot_raw_drift").min(), a13("pca10", "whitened_ot_raw_drift").max(), a13("pca20", "whitened_ot_raw_drift").min(), a13("pca20", "whitened_ot_raw_drift").max()],
    ["0.37", "0.58", "0.79", "0.96", "0.53", "0.79", "0.28", "0.43"])
row(S, "(J_ref; S15 Step 0; 4 test datasets × J_ref neighborhood 15/30)", "S15", "jref", "datasets; neighbourhoods", "values",
    lambda: [jref.dataset.nunique()] + sorted(jref.jref_neighbours.unique()), ["4", "15", "30"])
row(S, "norm ratio 0.02–0.07", "S15", "jref", "median_norm_J_over_norm_Jref", "range", lambda: jref.median_norm_J_over_norm_Jref.values, "0.02–0.07")
row(S, "median per-cell cosine −0.002 to 0.008", "S15", "jref", "median_per_cell_cosine", "range", lambda: jref.median_per_cell_cosine.values, "−0.002 to 0.008")
row(S, "per-cell λ₁ Spearman −0.107 to −0.005 across all four datasets (−0.018 to −0.005 on the three transition seeds)", "S15", "jref", "lambda1_spearman: all rows; v3_transition rows", "values",
    lambda: [jref.lambda1_spearman.min(), jref.lambda1_spearman.max(), jref[jref.dataset.str.startswith("v3_transition")].lambda1_spearman.min()], ["−0.107", "−0.005", "−0.018"])
row(S, "relative error after optimal scalar c* 1.00", "S15", "jref", "median_relative_error_cstar_J min and max", "values",
    lambda: [jref.median_relative_error_cstar_J.min(), jref.median_relative_error_cstar_J.max()], ["1.00", "1.00"])
row(S, "(median per-cell cosine ≈ 1.000)", "S15", "jref", "min basis_jref_fd_median_per_cell_cosine", "value", lambda: jref.basis_jref_fd_median_per_cell_cosine.min(), "1.000")
row(S, "Across S14 test rows the median relative Frobenius error is 1.000–1.005", "S14", "test", "jref_relative_frobenius_median, synthetic rows", "range",
    lambda: syn.jref_relative_frobenius_median.dropna().values, "1.000–1.005")
r15t = r15[r15.preset == "v3_transition"].sort_values("seed")
row(S, "peak-to-median ratio R of the 12-bin day-weighted strict-pseudotime λ₁ profile (reference, k = 15)", "S14", "test", "bins per profile; k", "values",
    lambda: [int(nbin.max()), r15.k.iloc[0]], ["12", "15"])
row(S, "on the three transition test seeds (1.91–2.69; |detected − true bin| 1, 0, 5)", "S14", "test",
    "reference k 15 v3_transition: seeds; R min/max; transition_abs_offset in seed order s2, s3, s4", "values",
    lambda: [len(r15t), r15t.transition_peak_to_median.min(), r15t.transition_peak_to_median.max()] + list(r15t.transition_abs_offset.values), ["3", "1.91", "2.69", "1", "0", "5"])
row(S, "and three no-fork seeds (0.58–1.44)", "S14", "test", "reference k 15 v3_nofork: seeds; R min/max", "values",
    lambda: [r15[r15.preset == "v3_nofork"].dataset.nunique(), r15[r15.preset == "v3_nofork"].transition_peak_to_median.min(), r15[r15.preset == "v3_nofork"].transition_peak_to_median.max()],
    ["3", "0.58", "1.44"])
row(S, "(dashed: 0.90 rule)", "S14b/S14c", "elig", "threshold 0.90 reproduces bootstrap flag (1 = yes)", "count",
    lambda: int(all((el[done].bootstrap_stability >= 0.9) == T(el[done].bootstrap_stability_at_least_90pct))), 1)
row(S, "S14b: 2 of 9 assessed seeds eligible (seeds 12, 16); seeds 13, 14, 15 could not be generated", "S14b", "elig",
    "S14b (s5–s16): eligible; completed; eligible seeds; failed-generation seeds", "values",
    lambda: [int(T(b14.eligible).sum()), int(b14.generation_status.eq("completed").sum())] + list(b14[T(b14.eligible)].seed) + list(b14[~b14.generation_status.eq("completed")].seed),
    ["2", "9", "12", "16", "13", "14", "15"])
row(S, "S14c (seeds 17–40): 14 seeds stopped at the generation memory guard; none of the 10 assessed was eligible", "S14c", "elig",
    "S14c seeds min/max; guard_stop; completed; eligible", "values",
    lambda: [c14.seed.min(), c14.seed.max(), int(c14.generation_status.eq("guard_stop").sum()), int(c14.generation_status.eq("completed").sum()), int(T(c14.eligible).sum())],
    ["17", "40", "14", "10", "0"])
row(S, "seeds with stable interior peaks failed only the ≥ 50-cell bin rule (minimum bins 42–46 cells)", "S14b/S14c", "elig", "seeds failing only the bin rule: min(bin_counts)", "range",
    lambda: el[only_bins].min_bin.values, "42–46", note="note: two of the three such seeds (8, 29, 36) are S14b s8 and S14c s29/s36")
row(S, "Four eligible seeds were required", "S14c", "g14c", "required_eligible_seeds", "count", lambda: int(g14c["required_eligible_seeds"]), 4)

S = "Legend Fig 4"
row(S, "Nine dyngen datasets (bifurcating seeds 10/16/18, cycle s0–s2, linear s0–s2; 5,000 cells, 1,000 genes)", "S21b", ["dsc", "dman"],
    "distinct datasets in scores.csv; bifurcating seeds; cells; genes", "values",
    lambda: [dsc.dataset.nunique()] + sorted(int(re.search(r"_s(\d+)_T8", d_).group(1)) for d_ in dsc[dsc.backbone == "bifurcating"].dataset.unique()) + [t8.cells.max(), t8.genes.max()],
    ["9", "10", "16", "18", "5000", "1000"])
row(S, "at eight synchronized time points", "S21b", "dpa", "T", "count", lambda: int(dpa["T"].max()), 8)
row(S, "MeasuredDrift drift seeds 0–2", "S21b", "dsc", "flow drift seeds min/max", "values", lambda: [fl.drift_seed.min(), fl.drift_seed.max()], ["0", "2"])
row(S, "flow J 1.00, 0.92–0.99 and 0.31–0.38 (inverted) on s10, s16, s18; score J 0.98, 0.58, 0.85", "S21b", "dsc", "flow min/min/max/min/max; score s10/s16/s18", "values",
    lambda: [fa.min()["bifurcating_v2_s10_T8"], fa.min()["bifurcating_v2_s16_T8"], fa.max()["bifurcating_v2_s16_T8"], fa.min()["bifurcating_v2_s18_T8"], fa.max()["bifurcating_v2_s18_T8"],
             scj["bifurcating_v2_s10_T8"], scj["bifurcating_v2_s16_T8"], scj["bifurcating_v2_s18_T8"]], ["1.00", "0.92", "0.99", "0.31", "0.38", "0.98", "0.58", "0.85"])
row(S, "Gate F1 (≥ 0.85) holds on 2 of 3 datasets (gate failed)", "S21b", "dsc", "bifurcations with min flow fork_auc >= 0.85; bifurcations", "values",
    lambda: [int((bif.groupby("dataset").fork_auc.min() >= 0.85).sum()), bif.dataset.nunique()], ["2", "3"])
mrot = lambda d_: fl[fl.dataset == d_].median_rotation
row(S, "cycle s1 0.19–0.32 and s2 0.36–0.40 vs linear 0.07–0.09; cycle s0 0.02 vs linear s0 0.08–0.10", "S21b", "dsc",
    "flow median_rotation min/max: cycle_s1, cycle_s2, linear s1+s2, cycle_s0 (min, max), linear_s0", "values",
    lambda: [mrot("cycle_s1_T8").min(), mrot("cycle_s1_T8").max(), mrot("cycle_s2_T8").min(), mrot("cycle_s2_T8").max(), lin.min(), lin.max(),
             mrot("cycle_s0_T8").min(), mrot("cycle_s0_T8").max(), mrot("linear_s0_T8").min(), mrot("linear_s0_T8").max()],
    ["0.19", "0.32", "0.36", "0.40", "0.07", "0.09", "0.02", "0.02", "0.08", "0.10"])
row(S, "(gate F2 failed: 2 of 3 cycles pass on every seed)", "S21b", "dsc", "cycles passing on every drift seed", "count", n_cycles, 2)
row(S, "Score J is symmetric (rotational index 0.00)", "S21b", "dsc", "max median_rotation, operator score", "value", lambda: dsc[dsc.operator == "score"].median_rotation.max(), "0.00")
row(S, "on linear backbones the fraction is 0 in the early and committed regions for every seed, and 0 over all 5,000 cells (maximum 0.0000 over 9 dataset × seed rows)",
    "S21b", ["dsc", "dmd"], "max fraction early/committed (scores.csv); max and row count of RESULT_S21b.md linear_all table (5000 cells)", "values",
    lambda: [float(fl[fl.backbone == "linear"][["fraction_positive_early", "fraction_positive_committed"]].values.max()), max(lin_all), max(lin_all), len(lin_all)],
    ["0", "0", "0.0000", "9"])
row(S, "s10 22 vs 329, s16 0 vs 746, s18 0 vs 776", "S21/S21b", ["d21", "dsc"], "n_fork T = 3 vs T = 8 per bifurcating dataset", "values",
    lambda: [t3["bifurcating_v2_s10"], t8n["bifurcating_v2_s10_T8"], t3["bifurcating_v2_s16"], t8n["bifurcating_v2_s16_T8"], t3["bifurcating_v2_s18"], t8n["bifurcating_v2_s18_T8"]],
    ["22", "329", "0", "746", "0", "776"])

S = "Legend Fig 5"
row(S, "(GSE140802; days 2/4/6); dense analyses use a fixed sample of 4,000 cells per day", "S14", "test", "kind larry n_cells / 3", "count",
    lambda: int(larry.loc["reference", "n_cells"] / 3), 4000, note="days are dataset design")
row(S, "Post hoc, reference: 1,000 permutations, one-sided p < 0.001 (null max 0.15); after centring each coordinate within an interval 0.50 (null max 0.04); population mean-shift predictor 0.19; OT displacement without a drift 0.54; λ₁ vs displacement magnitude −0.04", "S14 addendum", "add14", "as Results 5", "values",
    lambda: [add14["n_permutations"], add14["drift_pooled"]["p_one_sided"], add14["drift_pooled"]["null_max"], add14["drift_centred"]["spearman"], add14["drift_centred"]["null_max"], add14["mean_shift_pooled"]["spearman"], add14["ot_displacement_pooled"]["spearman"], add14["lambda1_vs_actual_magnitude"]], ["1000", "0.001", "0.15", "0.50", "0.04", "0.19", "0.54", "−0.04"])
row(S, "pooled over PCA coordinates of 554 barcoded cell–clone pairs, between the displacement predicted by integrating the drift over the 48-h interval", "S14 addendum", "add14", "n_cell_barcode_pairs; 48-h gap", "values", lambda: [add14["n_cell_barcode_pairs"], 48], ["554", "48"])
row(S, "95 % bootstrap intervals over cells (200 resamples): flow 0.66–0.73, mean displacement 0.85–0.89, no movement 0.83–0.94. Transport and drift were fitted on days 2 and 4 only", "S14 addendum", "add14", "ci95 flow, mean_displacement, no_movement; n_boot; train_days", "values",
    lambda: [95, add14["heldout_energy"]["n_boot"], *(add14["heldout_energy"]["ci95"][m][i] for m in ("flow", "mean_displacement", "no_movement") for i in (0, 1)), *add14["heldout_energy"]["train_days"]], ["95", "200", "0.66", "0.73", "0.85", "0.89", "0.83", "0.94", "2", "4"])
row(S, "vs five permutation nulls (●; pre-registered): reference 0.56 (null max 0.15), tuned 0.55 (null max 0.14)", "S14", "test",
    "larry: number of nulls; reference lineage_spearman, max null; selected lineage_spearman, max null", "values",
    lambda: [len(_json.loads(larry.loc["reference", "lineage_nulls"])), larry.loc["reference", "lineage_spearman"], max(_json.loads(larry.loc["reference", "lineage_nulls"])),
             larry.loc["selected", "lineage_spearman"], max(_json.loads(larry.loc["selected", "lineage_nulls"]))], ["5", "0.56", "0.15", "0.55", "0.14"])
row(S, "reference flow 0.69, tuned flow 0.94; mean-displacement baseline 0.87, no-movement baseline 0.88", "S14", "test", "larry energy distances", "values",
    lambda: [larry.loc["reference", "heldout_flow_energy"], larry.loc["selected", "heldout_flow_energy"], larry.loc["reference", "heldout_mean_displacement_energy"], larry.loc["reference", "heldout_no_movement_energy"]],
    ["0.69", "0.94", "0.87", "0.88"])
row(S, "S11 (pre-registered criteria; 6 of 14 met)", "S11", "s11md", "criterion table passes / rows", "values", lambda: [crit.count("True"), len(crit)], ["6", "14"])
nsc = len(va[(va.dataset == "scnt_hema") & (va.source_a == "labelling") & (va.source_b == "scvelo")])
row(S, "(n = 1,321 cells with a lineage velocity; median 0.69); scNT-seq hematopoiesis labeling vs Dynamo −0.05, labeling vs scVelo 0.03, scVelo vs Dynamo 0.15 (n = 1,947)", "S11", "va",
    "larry ot–lineage n and median; scnt_hema medians; scnt_hema n", "values",
    lambda: [len(vl), vl.median(), vsc("labelling", "dynamo_gene"), vsc("labelling", "scvelo"), vsc("scvelo", "dynamo_gene"), nsc],
    ["1321", "0.69", "−0.05", "0.03", "0.15", "1947"])
row(S, "LARRY cell-type median λ₁ of the OT and lineage flow J (PCA-30); Undifferentiated > Neutrophil and Monocyte for both", "S11", ["bct", "s11md"], "undiff > neu and mon for ot and lineage", "bool", undiff_top, True)
row(S, "Per-cell OT vs lineage λ₁ Spearman 0.27 (null level); OT flow J across fit seeds 1–2 and Dynamo SparseVFC 0.92–0.95", "S11", "oa",
    "independent_sources lambda1_spearman; ot vs ot measured_s1/s2/dynamo min/max", "values",
    lambda: [ind.lambda1_spearman, st.lambda1_spearman.min(), st.lambda1_spearman.max()], ["0.27", "0.92", "0.95"])
row(S, "S20 (pre-registered gates G1, G2 passed)", "S20", "s20md", "RESULT_S20.md gate table: G1 PASS and G2 PASS (1 = both)", "count",
    lambda: int(bool(re.search(r"\| G1 [^|]*\| PASS \|", s20md)) and bool(re.search(r"\| G2 [^|]*\| PASS \|", s20md))), 1)
row(S, "(2,000–16,000 per day and all 130,887 cells)", "S20", ["scal", "full"], "numeric n_per_day min/max; full n_cells", "values",
    lambda: [min(int(v) for v in scal.n_per_day if v != "full"), max(int(v) for v in scal.n_per_day if v != "full"), full.n_cells.sum()], ["2,000", "16,000", "130,887"])
nr(S, "mini-batch (4,000-cell batches)", "S20", CFG)
row(S, "low-rank (moscot rank 50/200) OT", "S20", "agr", "low-rank ranks", "values", lambda: sorted(int(s_.split("_r")[1]) for s_ in lr.solver.unique()), ["50", "200"])
row(S, "dense OT on the full data was not run (cost matrix ≈ 21 GB)", "S20", ["scal", "s20md"], "dense full rows; cost-matrix estimate", "values",
    lambda: [int(((scal.solver == "dense") & (scal.n_per_day == "full")).sum()), gb21()], ["0", "21"])
row(S, "Full LARRY with mini-batch OT: OT 291 s (2.54 GB), drift fit + Jacobians 1,278 s (1.31 GB)", "S20", "scal", "minibatch full: ot wall, rss/1e9; fit wall, rss/1e9", "values",
    lambda: [mo.loc["ot", "wall_seconds_external"], mo.loc["ot", "max_rss_bytes"] / 1e9, mo.loc["fit_and_jacobian", "wall_seconds_external"], mo.loc["fit_and_jacobian", "max_rss_bytes"] / 1e9],
    ["291", "2.54", "1,278", "1.31"])
row(S, "mini-batch median velocity cosine 0.9998 and λ₁ Spearman 0.9999 at 8,000 cells/day", "S20", "agr", "minibatch n_per_day 8000", "values",
    lambda: [a8.median_velocity_cosine, a8.flow_lambda1_spearman, a8.n_per_day], ["0.9998", "0.9999", "8,000"])
nr(S, "(the 4,000/day gate row is trivial: one batch = whole day)", "S20", "follows from the batch-size configuration constant (4,000-cell batches)")
row(S, "low-rank cosine 0.79–0.80", "S20", "agr", "lowrank median_velocity_cosine", "range", lambda: lr.median_velocity_cosine.values, "0.79–0.80")
row(S, "|λ₁ Spearman| ≤ 0.07", "S20", "agr", "max |flow_lambda1_spearman| over low-rank rows", "le", lambda: lr.flow_lambda1_spearman.abs().max(), "0.07")
row(S, "(cell-count-weighted mean of per-day medians, 11 cell types) vs the 4,000-per-day S11 result: Spearman 0.82", "S20", "full", "cell types; Spearman", "values",
    lambda: [full.cell_type.nunique(), ctrho()], ["11", "0.82"])

S = "Legend Fig 6"
row(S, "on four non-stationary v3 presets (PCA-20, 3 seeds each)", "S10", "ops", "J_score presets; seeds per preset", "values",
    lambda: [OPS[OPS.operator == "J_score"].preset.nunique(), OPS[OPS.operator == "J_score"].groupby("preset").seed.nunique().max()], ["4", "3"])
row(S, "Score-J AUC on default 0.16–0.31, stress 0.15–0.50 and batch 0.17–0.29", "S10", "ops", "J_score min/max default, stress, batch", "values",
    lambda: sum([[opauc("J_score", p).min(), opauc("J_score", p).max()] for p in ["v3_default", "v3_stress", "v3_batch"]], []), ["0.16", "0.31", "0.15", "0.50", "0.17", "0.29"])
row(S, "(AUC < 0.45, i.e. committed cells ranked above saddle cells, on 8 of 9 seeds), 0.43–0.52 on transition; OT flow J 0.97–1.00 on default", "S10", "ops",
    "J_score default/stress/batch rows < 0.45; rows; J_score transition min/max; J_flow_ot default min/max", "values",
    lambda: [int((sj < 0.45).sum()), len(sj), opauc("J_score", "v3_transition").min(), opauc("J_score", "v3_transition").max(), opauc("J_flow_ot").min(), opauc("J_flow_ot").max()],
    ["8", "9", "0.43", "0.52", "0.97", "1.00"])
row(S, "all fits on 10,691 non-targeting controls; 865 of 1,299 targets with ≥ 100 cells passed the BH-adjusted permutation test", "S18", ["pemb", "eff"],
    "n_controls; significant; eligible; minimum cells", "values",
    lambda: [pemb["n_controls"], eff["n_targets_significant"], eff["n_targets_eligible"], eff["minimum_cells"]], ["10,691", "865", "1,299", "100"])
row(S, "captured by m = 3 directions in control PCA-20: local score-J soft axes 0.045, global score J 0.038, local covariance 0.292, global covariance 0.589 (medians); random-subspace expectation 0.15",
    "S18", "cap", "k 20, m 3: m; k; medians of local_score_J, global_score_J, local_covariance, global_covariance; random_expectation", "values",
    lambda: [c20.m.iloc[0], c20.k.iloc[0], c20.local_score_J.median(), c20.global_score_J.median(), c20.local_covariance.median(), c20.global_covariance.median(), c20.random_expectation.iloc[0]],
    ["3", "20", "0.045", "0.038", "0.292", "0.589", "0.15"])
row(S, "P1 (fraction of targets above their random-null 95th percentile ≥ 0.70): 0.053", "S18", ["s18md", "cap"], "required (RESULT_S18.md); mean local_score_exceeds_null95", "values",
    lambda: [float(re.search(r"required >=([0-9]+\.[0-9]+)", s18md).group(1)), c20.local_score_exceeds_null95.mean()], ["0.70", "0.053"])
row(S, "k = 10/15/30 (report only): local score J 0.099/0.056/0.051", "S18", "cap", "m 3, k 10/15/30: median local_score_J", "values",
    lambda: [cap[(cap.k == k_) & (cap.m == 3)].local_score_J.median() for k_ in [10, 15, 30]], ["0.099", "0.056", "0.051"])
gH1 = re.search(r'"median_explained_fraction": ([0-9.]+), "required_median": ([0-9.]+), "n_exceeds_noise_p95": (\d+), "fraction_exceeds_noise_p95": ([0-9.]+)', s18bmd)
row(S, "(k = 20): median 0.31 (required ≥ 0.5); 235 of 865 targets (0.27) above the size-matched noise 95th percentile", "S18b", ["h1", "s18bmd"],
    "h1.csv k 20: k; median explained_fraction; required (RESULT_S18b.md); n exceeds_noise_p95; n targets; fraction", "values",
    lambda: [hh.k.iloc[0], hh.explained_fraction.median(), float(gH1.group(2)), int(T(hh.exceeds_noise_p95).sum()), len(hh), T(hh.exceeds_noise_p95).mean()],
    ["20", "0.31", "0.5", "235", "865", "0.27"])
row(S, "across 59 basins; partial Spearman ρ (ranks residualized on basin size and covariance trace) 0.05, one-sided p (negative) 0.65 (10,000 permutations); ordinary ρ 0.06; operator-free Leiden spread ρ 0.03 (p⁻ 0.60)",
    "S18b", "h2", "by_k 20: n_basins; partial rho, p; permutations; ordinary rho; leiden_ordinary rho, p", "values",
    lambda: [j20["n_basins"], j20["stiffness_partial"]["rho"], j20["stiffness_partial"]["p_one_sided_negative"], j20["stiffness_partial"]["permutations"],
             j20["stiffness_ordinary"]["rho"], j20["leiden_ordinary"]["rho"], j20["leiden_ordinary"]["p_one_sided_negative"]],
    ["59", "0.05", "0.65", "10,000", "0.06", "0.03", "0.60"])

S = "Legend Fig 7"
row(S, "operator-archetype view, K = 8)", "S15", "s15g", "k_selected operator view", "values", lambda: sorted(set(c["k_selected"] for c in c2.values())), ["8"])
row(S, "(descriptor view, K = 2)", "S15", "s15g", "k_selected descriptor view", "values", lambda: sorted(set(c["k_selected"] for c in c2d.values())), ["2"])
row(S, "on the three held-out v3_transition test seeds (s2–s4; PCA-15; S14 reference flow J)", "S15", "s15g", "test datasets in gate A1; k", "values",
    lambda: [len(oa1), sorted(set(d["k"] for d in oa1.values()))[0]], ["3", "15"])
row(S, "(bars: drift seed 0, the gate rows; dots: drift seeds 1–2, report only)", "S15", ["s15g", "rr"], "gate-row drift seed; other drift seeds in regime_recovery.csv (flow, saddle, k 15, test seeds)", "values",
    lambda: [sorted(set(d["drift_seed"] for d in oa1.values()))[0], *sorted(set(rr[(rr.regime == "saddle") & (rr.operator == "flow") & (rr.k == 15) & rr.dataset.isin(TESTS)].drift_seed) - {0})],
    ["0", "1", "2"])
row(S, "Operator view: recall 0.72–0.79, precision 3.4–4.1× base rate (A1 passes); descriptor view: recall 0.88–0.95, precision 1.5–3.2× (fails on one seed)", "S15", "s15g",
    "gate A1 recall and precision_over_base_rate min/max per view; descriptor seeds below 3×", "values",
    lambda: [min(d["recall"] for d in oa1.values()), max(d["recall"] for d in oa1.values()), min(d["precision_over_base_rate"] for d in oa1.values()), max(d["precision_over_base_rate"] for d in oa1.values()),
             int(A1["operator_archetype"]["pass"]), min(d["recall"] for d in dk1.values()), max(d["recall"] for d in dk1.values()),
             min(d["precision_over_base_rate"] for d in dk1.values()), max(d["precision_over_base_rate"] for d in dk1.values()), sum(d["precision_over_base_rate"] < 3 for d in dk1.values())],
    ["0.72", "0.79", "3.4", "4.1", "1", "0.88", "0.95", "1.5", "3.2", "1"])
row(S, "Dashed lines: gate thresholds (recall ≥ 0.6, precision ≥ 3× base rate)", "S15", "s15g", "thresholds reproduce A1 flags (1 = yes)", "count",
    lambda: int(all(all(d["recall"] >= 0.6 and d["precision_over_base_rate"] >= 3 for d in A1[v]["saddle"]) == A1[v]["pass"] for v in ["operator_archetype", "descriptor_kmeans"])), 1)
mm2 = lambda C, key: [min(c[key] for c in C.values()), max(c[key] for c in C.values())]
row(S, "Operator view: flow 0.35–0.39, Leiden 0.35–0.37, score 0.05–0.31; flow beats score on 3 of 3 seeds and Leiden on 1 of 3 (A2 fails)", "S15", "s15g",
    "gate A2 operator view min/max nmi_flow, leiden_nmi, nmi_score; seeds flow>score; seeds flow>Leiden; pass", "values",
    lambda: mm2(c2, "nmi_flow") + mm2(c2, "leiden_nmi") + mm2(c2, "nmi_score") + [sum(c["nmi_flow"] > c["nmi_score"] for c in c2.values()), len(c2),
             sum(c["nmi_flow"] > c["leiden_nmi"] for c in c2.values()), len(c2), int(A2["operator_archetype"]["pass"])],
    ["0.35", "0.39", "0.35", "0.37", "0.05", "0.31", "3", "3", "1", "3", "0"])
row(S, "Descriptor view: flow 0.06–0.23, Leiden 0.014–0.017, score 0.04–0.07", "S15", "s15g", "gate A2 descriptor view min/max nmi_flow, leiden_nmi, nmi_score", "values",
    lambda: mm2(c2d, "nmi_flow") + mm2(c2d, "leiden_nmi") + mm2(c2d, "nmi_score"), ["0.06", "0.23", "0.014", "0.017", "0.04", "0.07"])
row(S, "between drift-fit seeds 0, 1, 2 (9 pairs per view): operator view 0.22–0.44, descriptor view −0.02 to 0.53 (gate ≥ 0.5 for every pair; fails)", "S15", "s15g",
    "gate A3 pairs per view; operator min/max; descriptor min/max; any view passing", "values",
    lambda: [len(a3o), len(a3d), min(a3o), max(a3o), min(a3d), max(a3d), int(A3["operator_archetype"]["pass"] or A3["descriptor_kmeans"]["pass"])],
    ["9", "9", "0.22", "0.44", "−0.02", "0.53", "0"])

S = "Legend Fig S1"
dd = diff[diff.preset == "v3_default"]
rat = lambda est, var, meth="pca": dd[(dd.estimator == est) & (dd.variant == var) & (dd.method == meth) & (dd.quantity == "trace/d")].ratio_to_truth_all
row(S, "v3_default, seeds 0–2", "S10", "diff", "seeds of v3_default rows", "values", lambda: [dd.seed.min(), dd.seed.max()], ["0", "2"])
row(S, "(truth from 32 forward paths per cell, Δt = 48 h for clone and OT estimators, 6 h for the score–flow estimator)", "S10", "diff",
    "horizon_hours per estimator: clone/OT; score_flow (the 32 paths are not recorded)", "values",
    lambda: [dd[dd.estimator.isin(["clone_dispersion", "ot_noise"])].horizon_hours.max(), dd[dd.estimator == "score_flow"].horizon_hours.max()], ["48", "6"],
    note="'32 forward paths' not recorded in a result file")
row(S, "clone dispersion −1.88 to −1.59 (PCA), OT noise −1.25 to −1.09; uncorrected: clone −0.01 to 0.18, OT 0.47–0.50", "S10", "diff",
    "ratio_to_truth_all (trace/d, PCA) min/max: clone literal; OT minus Σ_meas; clone uncorrected; OT uncorrected", "values",
    lambda: [rat("clone_dispersion", "literal").min(), rat("clone_dispersion", "literal").max(), rat("ot_noise", "minus_trace_sigma_meas_over_d").min(), rat("ot_noise", "minus_trace_sigma_meas_over_d").max(),
             rat("clone_dispersion", "uncorrected").min(), rat("clone_dispersion", "uncorrected").max(), rat("ot_noise", "uncorrected").min(), rat("ot_noise", "uncorrected").max()],
    ["−1.88", "−1.59", "−1.25", "−1.09", "−0.01", "0.18", "0.47", "0.50"])
row(S, "score–flow (raw labeling) 0.14–0.20", "S10", "diff", "score_flow raw_shared PCA ratio min/max", "range", lambda: rat("score_flow", "raw_shared").values, "0.14–0.20")
row(S, "exceeds the 48-h effective diffusion", "S10", ["dsum", "diff"], "Σ_meas > 48-h truth for every seed, PCA and FA; horizon of the truth (h)", "values",
    lambda: [int(all(a > b for m_ in ["pca", "fa"] for a, b in zip(sm_(m_), tr48(m_)))), dd[dd.estimator == "clone_dispersion"].horizon_hours.max()], ["1", "48"])
row(S, "Green band: within a factor of 2", "S10", "dsum", "gate definition contains 'within x2'", "bool", lambda: "within x2" in dsum["gate_diffusion"]["definition"], True)
row(S, "PCA 0.152–0.171 vs 0.088–0.105; FA 0.033–0.037 vs 0.0059–0.0066", "S10", ["dsum", "diff"], "Σ_meas trace/d min/max PCA; 48-h truth min/max PCA; same FA", "values",
    lambda: [min(sm_("pca")), max(sm_("pca")), min(tr48("pca")), max(tr48("pca")), min(sm_("fa")), max(sm_("fa")), min(tr48("fa")), max(tr48("fa"))],
    ["0.152", "0.171", "0.088", "0.105", "0.033", "0.037", "0.0059", "0.0066"])
scc = lambda var: dd[(dd.estimator == "score_flow") & (dd.variant == var) & (dd.method == "pca") & (dd.quantity == "trace/d")].spearman_committed
row(S, "(gate ≥ 0.5): raw shared γ 0.052–0.115; smoothed shared γ 0.015–0.295", "S10", ["dsum", "diff"], "gate definition '>= 0.5'; spearman_committed min/max raw_shared, smoothed_shared", "values",
    lambda: [0.5 if "Spearman >= 0.5" in dsum["gate_diffusion"]["definition"] else np.nan, scc("raw_shared").min(), scc("raw_shared").max(), scc("smoothed_shared").min(), scc("smoothed_shared").max()],
    ["0.5", "0.052", "0.115", "0.015", "0.295"])

S = "Legend Fig 6"
row(S, "S22 (score-J stiffness vs LARRY sister-cell fate coherence; pre-registered, all gates failed) and S18d (RPE1 dose-response confirmation; underpowered)", "S22/S18d", ["s22sum", "dg"],
    "S22 gates failed (count of 3); S18d outcome underpowered (1 = yes)", "values",
    lambda: [sum(not s22sum[k_]["passed"] for k_ in ["gate_L1_pooled", "gate_L2_undifferentiated", "gate_L3_baselines"]), int(dg["outcome"] == "underpowered")], ["3", "1"])
row(S, "(filled: seed 0, gated; open: seeds 1–2, report only)", "S22", "s22", "gated rows are seed 0 only; seeds present", "values",
    lambda: [int(s22[s22.gated].seed.unique().tolist() == [0]), s22.seed.min(), s22.seed.max()], ["1", "0", "2"])
row(S, "Pooled (1,818 clone/time pairs, 67 with tied modal types): λ₁ ρ −0.003 (one-sided permutation p 1); Undifferentiated (1,401 pairs): λ₁ ρ −0.10 (p 1), centroid distance 0.22, density 0.08; partial ρ of λ₁ given density −0.08",
    "S22", "s22", "seed 0 pooled n_pairs, n_tied, rho, p; undifferentiated n_pairs, rho, p, rho_centroid, rho_density, partial", "values",
    lambda: [g22.loc["pooled", "n_pairs"], g22.loc["pooled", "n_tied_pairs"], g22.loc["pooled", "rho_lambda1_H"], g22.loc["pooled", "p_permutation_positive"],
             g22.loc["undifferentiated", "n_pairs"], g22.loc["undifferentiated", "rho_lambda1_H"], g22.loc["undifferentiated", "p_permutation_positive"],
             g22.loc["undifferentiated", "rho_centroid_distance_H"], g22.loc["undifferentiated", "rho_density_H"], g22.loc["undifferentiated", "partial_rho_given_density"]],
    ["1,818", "67", "−0.003", "1", "1,401", "−0.10", "1", "0.22", "0.08", "−0.08"])
row(S, "on day *t* and the entropy H of its sister cells' types on day *t* + 2", "S22", "s22cl", "clones.csv t values; sampled days (RESULT_S22.md): offset to the next sampled day", "values",
    lambda: (lambda days: sorted({days[days.index(t_) + 1] - t_ for t_ in s22cl.t.unique()}))([int(x) for x in re.search(r"each on days (\d+), (\d+) and (\d+)", s22md).groups()]),
    ["2"], note="next sampled day − t for every t in clones.csv, sampled days from RESULT_S22.md")
_lam = s22[s22.cohort.isin(["pooled", "undifferentiated"])].rho_lambda1_H
row(S, "λ₁ ρ over seeds 0–2 and both cohorts: −0.130 to −0.003", "S22", "s22", "rho_lambda1_H, cohorts pooled + undifferentiated, seeds 0–2", "range", lambda: _lam.values, "−0.130 to −0.003")
row(S, "95th percentile of the across-T median under 10,000 covariance-matched random subspaces", "S18d", "dcj_k562_N1", "capture_*.json null_definition count (all four identical)", "values",
    lambda: [95 if "null_across_T_median_p95" in dg["comparisons"]["k562"]["N1"]["weak"] else np.nan,
             (lambda s_: int(re.match(r"(\d+) across-T medians", s_.pop()).group(1)) if len(s_) == 1 else np.nan)({d_["null_definition"] for d_ in dcj.values()})], ["95", "10,000"])
_w = lambda s_, n_: dg["comparisons"][s_][n_]["weak"]
row(S, "RPE1 (pre-registered; underpowered: 1 qualifying T target, ≥ 10 required): score J 0.26/0.27 (N1/N2), covariance 0.13/0.62, null95 0.44/0.64", "S18d", "dg",
    "gates.json rpe1 weak N1/N2: n_T, threshold, median_S_J_T, median_S_C_T, null95", "values",
    lambda: [dg["n_T"]["rpe1"], dg["underpowered_threshold_n_T"], _w("rpe1", "N1")["median_S_J_T"], _w("rpe1", "N2")["median_S_J_T"], _w("rpe1", "N1")["median_S_C_T"], _w("rpe1", "N2")["median_S_C_T"],
             _w("rpe1", "N1")["null_across_T_median_p95"], _w("rpe1", "N2")["null_across_T_median_p95"]], ["1", "10", "0.26", "0.27", "0.13", "0.62", "0.44", "0.64"])
row(S, "K562 (report only; 14 T, 38 M targets): score J 0.42/0.34, covariance 0.35/0.20, null95 0.44/0.39; score J above covariance, one-sided paired Wilcoxon p 0.27/0.09; T above mitochondrial/translation (M) targets, one-sided Mann–Whitney p 0.002/0.012",
    "S18d", "dg", "gates.json k562 weak N1/N2: n_T, n_M, medians, null95, wilcoxon p, mann-whitney p", "values",
    lambda: [dg["n_T"]["k562"], dg["n_M"]["k562"], _w("k562", "N1")["median_S_J_T"], _w("k562", "N2")["median_S_J_T"], _w("k562", "N1")["median_S_C_T"], _w("k562", "N2")["median_S_C_T"],
             _w("k562", "N1")["null_across_T_median_p95"], _w("k562", "N2")["null_across_T_median_p95"], _w("k562", "N1")["wilcoxon"]["p_one_sided_greater"], _w("k562", "N2")["wilcoxon"]["p_one_sided_greater"],
             _w("k562", "N1")["mannwhitney"]["p_one_sided_greater"], _w("k562", "N2")["mannwhitney"]["p_one_sided_greater"]],
    ["14", "38", "0.42", "0.34", "0.35", "0.20", "0.44", "0.39", "0.27", "0.09", "0.002", "0.012"])

S = "Supplementary tables"
row(S, "per seed (0–2) and cohort (pooled, Undifferentiated, pooled without tied pairs, day 2 → 4, day 4 → 6)", "S22", "s22", "seeds; t values of the per-day cohorts (t2, t4) and t + 2", "values",
    lambda: [s22.seed.min(), s22.seed.max(), 2 if "t2" in set(s22.cohort) else np.nan, 4, 4 if "t4" in set(s22.cohort) else np.nan, 6], ["0", "2", "2", "4", "4", "6"])
row(S, "gated rows (seed 0) and gate outcomes (L1–L3 all failed)", "S22", "s22sum", "gated seed; gates failed (count of 3)", "values",
    lambda: [s22[s22.gated].seed.max(), sum(not s22sum[k_]["passed"] for k_ in ["gate_L1_pooled", "gate_L2_undifferentiated", "gate_L3_baselines"])], ["0", "3"])
row(S, "per screen (RPE1 pre-registered, K562 report only)", "S18d", "dg", "primary_screen == rpe1 (1 = yes)", "count", lambda: int(dg["primary_screen"] == "rpe1"), 1)
row(S, "overall outcome underpowered", "S18d", "dg", "gates.json outcome", "bool", lambda: dg["outcome"] == "underpowered", True)

S = "Legend Fig S2"
row(S, "synchronized sampling at T = 3 (S21) or T = 8 (S21b); flow J drift seeds 0–2", "S21/S21b", ["dpa", "d21"], "S21b T; S21 drift seeds min/max (T = 3 not recorded)", "values",
    lambda: [dpa["T"].max(), d21[d21.operator == "flow"].drift_seed.min(), d21[d21.operator == "flow"].drift_seed.max()], ["8", "0", "2"], note="T = 3 for S21 not in a committed result table")
row(S, "Cells in the fork region at T = 3: 22, 0, 0 (s10, s16, s18; F1 not evaluable for s16 and s18) vs 329, 746, 776 at T = 8", "S21/S21b", ["d21", "dsc"], "n_fork T3 then T8", "values",
    lambda: [t3["bifurcating_v2_s10"], t3["bifurcating_v2_s16"], t3["bifurcating_v2_s18"], t8n["bifurcating_v2_s10_T8"], t8n["bifurcating_v2_s16_T8"], t8n["bifurcating_v2_s18_T8"]],
    ["22", "0", "0", "329", "746", "776"])
f3 = d21[d21.operator == "flow"].copy(); f3["pair"] = f3.dataset.str.extract(r"_(s\d)$")[0]
def cyc_pass(f, p):
    c = f[(f.backbone == "cycle") & (f.pair == p)].sort_values("drift_seed").median_rotation.values
    l = f[(f.backbone == "linear") & (f.pair == p)].sort_values("drift_seed").median_rotation.values
    return bool(np.all(c > l))
row(S, "cycle above linear on all drift seeds at T = 3 for s0/s1/s2: True, False, True; at T = 8: False, True, True", "S21/S21b", ["d21", "dsc"], "cycle > linear on all seeds, s0/s1/s2, T3 then T8", "bool",
    lambda: [cyc_pass(f3, p) for p in ["s0", "s1", "s2"]] + [cyc_pass(fl, p) for p in ["s0", "s1", "s2"]], [True, False, True, False, True, True])
row(S, "1.00 at T = 3 on every dataset and seed; at T = 8, s10 0.02–0.06, s16 1.00, s18 0.00", "S21/S21b", ["d21", "dsc"], "T3 min/max early fraction; T8 s10 min/max, s16 min, s18 max", "values",
    lambda: [e3.min(), e3.max(), fe.min()["bifurcating_v2_s10_T8"], fe.max()["bifurcating_v2_s10_T8"], fe.min()["bifurcating_v2_s16_T8"], fe.max()["bifurcating_v2_s18_T8"]],
    ["1.00", "1.00", "0.02", "0.06", "1.00", "0.00"])

S = "Legend Fig S3"
row(S, "on v3_default and v3_transition seeds 0–2 for PCA-10/20/50 and FA-20, each with four variants", "S13", "s13", "seeds min/max; variants (incl. raw_s12) per embedding", "values",
    lambda: [s13.seed.min(), s13.seed.max(), s13.groupby("embedding").variant.nunique().max()], ["0", "2", "4"])
def _dmax():
    out = 0.0
    for e in ["pca10", "pca20", "pca50", "fa20"]:
        r13 = s13[(s13.embedding == e) & (s13.variant == "raw_s12")].set_index("dataset").auc_lam1
        r12 = s12[s12.embedding == e].assign(dataset=lambda x: x.preset + "_s" + x.seed.astype(str)).set_index("dataset").auc_lam1
        out = max(out, float((r13 - r12.reindex(r13.index)).abs().max()))
    return out
row(S, "(max |ΔAUC| over PCA-10/20/50/FA-20: 0.0000)", "S12/S13", ["s13", "s12"], "max |S13 raw_s12 − S12| AUC", "value", _dmax, "0.0000")

S = "Legend Fig S4"
row(S, "(S11, pre-registered; 6 of 14 criteria met)", "S11", "s11md", "criterion table", "values", lambda: [crit.count("True"), len(crit)], ["6", "14"])
row(S, "PCA-30, 4,000 cells/day", "S11", ["s11md", "test"], "RESULT_S11.md PCA-30; S14 LARRY n_cells/3", "values",
    lambda: [30 if "PCA-30" in s11md else np.nan, larry.loc["reference", "n_cells"] / 3], ["30", "4,000"])
row(S, "gray ticks: five permutation nulls", "S11", "nu", "null rows per larry independent_sources block", "count", lambda: len(nl), 5)
row(S, "LARRY OT vs lineage: λ₁ Spearman 0.27 (null max 0.27), |cos| 0.29 (null max 0.32)", "S11", ["oa", "nu"], "as Results 5", "values",
    lambda: [ind.lambda1_spearman, nl.lambda1_spearman.max(), ind.leading_eigenvector_abs_cosine_median, nl.leading_eigenvector_abs_cosine_median.max()], ["0.27", "0.27", "0.29", "0.32"])


## Coverage audit: numeric tokens in the manuscript sections that are not inside a checked snippet

In [14]:
MASKS = [r"\[\[.*?\]\]", r"\$\$.*?\$\$", r"\[\d+(?:\s*[,–]\s*\d+)*\]",
         r"(?:v56 )?(?:Supplementary )?Fig(?:ure)?s?\.?\s+S?\d+[a-z]?(?:\s*(?:,|–|and)\s*S?\d*[a-z]?(?![\w]))*",
         r"Supplementary (?:Table|Note)s? S?\d+(?:–S?\d+)?", r"\bTable \d+", r"\bResults \d(?:(?:, | and |–)\d)*", r"\bStep \d",
         r"\bscJDO v1\b", r"\bv\d+\b"]
def mask(txt):
    for p in MASKS:
        txt = re.sub(p, lambda m: " " * len(m.group(0)), txt, flags=re.S)
    return txt
TOK = re.compile(r"(?<![\w.\-−/])[−\-+]?\d+(?:,\d{3})*(?:\.\d+)?(?:e-\d+)?(?![\w])")
def covered(sec):
    spans = []
    for r_ in ROWS:
        if r_["section"] == sec:
            for m in re.finditer(re.escape(r_["quoted"]), SEC[sec]):
                spans.append((m.start(), m.end()))
    return spans


In [15]:
NONCLAIM = {   # (section, token context) -> reason; identifiers and labels, not numeric claims
    "seed / dataset identifier": [r"seeds? \d+(?:–\d+)?", r"seeds \d+, \d+, \d+", r"seeds \d+/\d+/\d+", r"seeds 0–2", r"seed 0", r"seed 42", r"bifurcating seeds 10/16/18",
                                  r"drift seeds 0, 1, 2", r"drift seeds 1–2", r"fit seeds 1–2", r"seeds \d+–\d+"],
    "panel / gate / task label": [r"gates? [A-Z]\d(?:–[A-Z]\d)?", r"\bH[12]\b", r"\bP1\b"],
    "date / accession / version identifier": [r"2026-10-\d\d", r"GSE\d+", r"default_rng\(0\)", r"\bv3_\w+"],
    "operator-space / math notation": [r"−β∇²E", r"A⁻ᵀHA⁻¹", r"Re λ₁ > 0", r"Re > 0", r"Re λ₁/max\|λ\|", r"\|λ\|", r"> 0\.5", r"< 0\.5", r"k\*"],
    "dimension in a name (PCA-k etc.)": [r"PCA-\d+", r"FA-\d+", r"scVI-\d+", r"PCA-10/20/50"],
    "list / box numbering, method names": [r"(?m)^\d\. ", r"Box \d", r"CellRank 2", r"\*t − 1\*"],
    "sampling-day labels (design)": [r"from day 4", r"held-out day-6", r"held-out day 6"],
    "formula": [r"E\(X, Y\) = 2 E", r"λ₁ > 0"],
    "accession identifier": [r"figshare \d+"],
    "statement citing v56 by version (not a v57 number)": [r"version 56", r"in v56, a shuffle test at 100,000 cells"],
}
def audit():
    out = []
    for sec in [s_ for s_ in SEC if s_ not in ("Methods: Pre-registration and review",)]:
        txt = SEC[sec]; mk = mask(txt); spans = covered(sec)
        for p_list in NONCLAIM.values():
            for p in p_list:
                spans += [(m.start(), m.end()) for m in re.finditer(p, txt)]
        for m in TOK.finditer(mk):
            if not any(s0 <= m.start() and m.end() <= s1 for s0, s1 in spans):
                out.append(dict(section=sec, token=m.group(0), context=txt[max(0, m.start() - 50):m.end() + 30].replace("\n", " ")))
    return pd.DataFrame(out, columns=["section", "token", "context"])
UNCOV = audit()
print(len(UNCOV), "uncovered numeric tokens")
print(UNCOV.to_string() if len(UNCOV) else "")

0 uncovered numeric tokens



## Internal consistency of the final manuscript
(a) callouts → figure files and legends; (b) citations ↔ reference list; (c) numbers shared by text and legend.

In [16]:
PROBLEMS = []   # (check, item, detail)
leg_paras = {k[len("Legend Fig "):]: v for k, v in SEC.items() if k.startswith("Legend Fig ")}
def panels_of(legend):
    ps = set()
    for m in re.finditer(r"\*\*([a-h])(?:[–-]([a-h]))?\*\*", legend):
        a_, b_ = m.group(1), m.group(2) or m.group(1)
        ps |= {chr(c) for c in range(ord(a_), ord(b_) + 1)}
    return ps
def expand_panels(s_):
    out = set()
    for part in re.split(r",", s_ or ""):
        part = part.strip()
        if re.fullmatch(r"[a-h]–[a-h]", part):
            out |= {chr(c) for c in range(ord(part[0]), ord(part[2]) + 1)}
        elif re.fullmatch(r"[a-h]", part):
            out.add(part)
    return out
callouts = []
scan = re.sub(r"\[\[.*?\]\]", lambda m: " " * len(m.group(0)), main_txt + "\n" + leg_txt, flags=re.S)
for m in re.finditer(r"(v56 )?(Supplementary )?(Fig(?:ure)?s?)\.?\s+((?:S?\d+[a-z]?(?:[,–][a-h])*)(?:\s*(?:,|–|and)\s*S?\d+[a-z]?(?:[,–][a-h])*)*)", scan):
    if m.group(1):           # v56 numbering (provenance statement), not a v57 callout
        continue
    supp = bool(m.group(2))
    for item in re.finditer(r"(S?)(\d+)([a-h](?:[,–][a-h])*)?", m.group(4)):
        pass
    toks = re.findall(r"(S?\d+)([a-h](?:(?:,|–)[a-h])*)?", m.group(4))
    nums_ = [t[0] for t in toks]
    if re.search(r"S?\d+\s*–\s*S?\d+", m.group(4)) and len(nums_) == 2 and not re.search(r"[a-h]–", m.group(4)):
        a_, b_ = int(nums_[0].lstrip("S")), int(nums_[1].lstrip("S"))
        toks = [(("S" if nums_[0].startswith("S") else "") + str(i), "") for i in range(a_, b_ + 1)]
    for n_, pan in toks:
        if supp and not n_.startswith("S"):
            n_ = "S" + n_
        if not supp and n_.startswith("S"):
            supp = True
        callouts.append(dict(fig=n_, panels=pan, where=scan[max(0, m.start() - 30):m.end() + 10].replace("\n", " ")))
CO = pd.DataFrame(callouts)
for fig_, g_ in CO.groupby("fig"):
    kind_ = "supplementary" if fig_.startswith("S") else "main"
    fpath = resolve(f"manuscript/figures/{kind_}/Fig{fig_}.pdf")
    if not fpath.exists():
        PROBLEMS.append(("callout → file", f"Fig {fig_}", f"no file manuscript/figures/{kind_}/Fig{fig_}.pdf"))
    if fig_ not in leg_paras:
        PROBLEMS.append(("callout → legend", f"Fig {fig_}", "no legend paragraph in the final manuscript"))
        continue
    have = panels_of(leg_paras[fig_])
    for _, c_ in g_.iterrows():
        miss = expand_panels(c_.panels.replace("–", "–")) - have if c_.panels else set()
        if miss:
            PROBLEMS.append(("callout → panel", f"Fig {fig_}{c_.panels}", f"panel(s) {sorted(miss)} not in the legend (legend panels {sorted(have)}); at: …{c_.where}…"))
# figure files / legends without a callout or legend
for kind_, pre in [("main", ""), ("supplementary", "S")]:
    for p in sorted(resolve(f"manuscript/figures/{kind_}").glob("Fig*.pdf")):
        n_ = p.stem[3:]
        if n_ not in set(CO.fig):
            PROBLEMS.append(("file without callout", p.stem, f"manuscript/figures/{kind_}/{p.name} is never called out in the final manuscript"))
        if n_ not in leg_paras:
            PROBLEMS.append(("file without legend", p.stem, f"manuscript/figures/{kind_}/{p.name} has no legend in the final manuscript"))
# tables and supplementary notes
for m in re.finditer(r"(v56 )?Supplementary (Table|Note) (S?\d+)", scan):
    v56_, kind_, n_ = m.groups()
    if v56_:
        continue
    if kind_ == "Note":
        if not re.search(rf"Supplementary Note {n_}\b\s*[—-]", main_txt):
            PROBLEMS.append(("callout → supplementary note", f"Supplementary Note {n_}", f"called out but no Supplementary Note {n_} section exists in the final manuscript; at: …{scan[max(0, m.start() - 40):m.end() + 5]}…"))
    else:
        entry = re.search(rf"\*\*Supplementary Table {n_}\*\* — (.*)", SEC["Supplementary tables"])
        if not entry:
            PROBLEMS.append(("callout → supplementary table", f"Supplementary Table {n_}", "no entry in the Supplementary tables list"))
        elif "TODO" in entry.group(1):
            PROBLEMS.append(("callout → supplementary table", f"Supplementary Table {n_}", "entry is a TODO placeholder: " + entry.group(1).strip()))
PROBLEMS = list(dict.fromkeys(PROBLEMS))
if not re.search(r"\*\*Table 1 \|", main_txt):
    PROBLEMS.append(("callout → table", "Table 1", "no Table 1 in the manuscript"))
# (b) citations
def cites(txt):
    out = []
    for m in re.finditer(r"\[(\d+(?:\s*[,–]\s*\d+)*)\]", txt):
        if re.search(r"∈\s*$", txt[max(0, m.start() - 3):m.start()]) or re.search(r"(?<![\d.])0(?![\d.])", m.group(1)):
            continue
        for part in m.group(1).split(","):
            part = part.strip()
            if "–" in part:
                a_, b_ = map(int, part.split("–")); out += list(range(a_, b_ + 1))
            else:
                out.append(int(part))
    return out
cited = cites(body)
reflist = [int(m.group(1)) for m in re.finditer(r"^(\d+)\. ", refs_txt, re.M)]
CIT = dict(n_refs=len(reflist), refs_consecutive=reflist == list(range(1, len(reflist) + 1)),
           cited_missing=sorted(set(cited) - set(reflist)), uncited=sorted(set(reflist) - set(cited)))
first = list(dict.fromkeys(cited))
CIT["out_of_order_first_citations"] = [(n_, [m_ for m_ in first[i + 1:] if m_ < n_]) for i, n_ in enumerate(first) if any(m_ < n_ for m_ in first[i + 1:])]
for n_ in CIT["cited_missing"]:
    PROBLEMS.append(("citation", f"[{n_}]", "cited in the text but not in the reference list"))
for n_ in CIT["uncited"]:
    PROBLEMS.append(("citation", f"ref {n_}", "in the reference list but never cited"))
print(CIT)
print(pd.DataFrame(PROBLEMS, columns=["check", "item", "detail"]).to_string())

{'n_refs': 57, 'refs_consecutive': True, 'cited_missing': [], 'uncited': [], 'out_of_order_first_citations': []}
Empty DataFrame
Columns: [check, item, detail]
Index: []


### (c) Numbers that appear in both the main text and a legend
Each pair names the same quantity; the two quoted values agree if they are equal at the coarser quoted precision
(percent converted to fractions).

In [17]:
TL = [  # quantity, text section, text quote, text values, legend section, legend quote, legend values
    ("C5 zero fraction, v1/v2", "Results 2", "36–37 % in v1 and v2 data", ["0.36", "0.37"], "Legend Fig 2", "v1 median 0.36, v2 0.37", ["0.36", "0.37"]),
    ("C5 zero fraction, real range", "Results 2", "84–96 % in real data", ["0.84", "0.96"], "Legend Fig 2", "real 0.84–0.96", ["0.84", "0.96"]),
    ("match rates v3/v1/v2", "Results 2", "Its overall match rate was 0.39 (v1 0.35, v2 0.22", ["0.39", "0.35", "0.22"], "Legend Fig 2", "v1 0.35, v2_default 0.22, v3_default 0.39", ["0.39", "0.35", "0.22"]),
    ("dyngen match rate", "Results 2", "at 0.51 (range 0.44–0.60)", ["0.51", "0.44", "0.60"], "Legend Fig 2", "0.51 (range 0.44–0.60)", ["0.51", "0.44", "0.60"]),
    ("S09 targets met", "Results 2", "met 6 of 13 on each seed", ["6", "13"], "Legend Fig 2", "met 6 of 13 on each seed", ["6", "13"]),
    ("S14 reference AUC", "Results 3", "AUC 0.89–0.97 across k = 10–20 principal components, and 0.97–1.00", ["0.89", "0.97", "0.97", "1.00"],
     "Legend Fig 3", "AUC 0.89–0.97 on v3_transition and 0.97–1.00 on the other presets", ["0.89", "0.97", "0.97", "1.00"]),
    ("S14 tuned AUC", "Results 3", "(0.42–0.95)", ["0.42", "0.95"], "Legend Fig 3", "0.42–0.95 on v3_transition", ["0.42", "0.95"]),
    ("S10 v3_default AUCs", "Results 3", "fork AUC 0.97–1.00, whereas flow operators from labeling velocity (0.53–0.89) or scVelo velocity (0.27–0.51) were weaker. The score operator ranked committed cells above saddle cells (0.16–0.31)",
     ["0.97", "1.00", "0.53", "0.89", "0.27", "0.51", "0.16", "0.31"], "Legend Fig 3", "OT flow J 0.97–1.00, labeling flow J 0.53–0.89 (raw and smoothed variants), scVelo flow J 0.27–0.51, score J 0.16–0.31",
     ["0.97", "1.00", "0.53", "0.89", "0.27", "0.51", "0.16", "0.31"]),
    ("S12 embeddings", "Results 3", "PCA-10 (0.88–1.00) and PCA-20 (0.82–1.00) and degraded in PCA-50 (0.56–0.97), scVI-10 [45] (0.31–0.96) and Factor Analysis (FA-20; 0.04–0.53",
     ["0.88", "1.00", "0.82", "1.00", "0.56", "0.97", "0.31", "0.96", "0.04", "0.53"], "Legend Fig 3", "PCA-10 0.88–1.00, PCA-20 0.82–1.00, PCA-50 0.56–0.97, scVI-10 0.31–0.96, FA-20 0.04–0.53",
     ["0.88", "1.00", "0.82", "1.00", "0.56", "0.97", "0.31", "0.96", "0.04", "0.53"]),
    ("S13 whitening", "Results 3", "(PCA-20 0.37–0.58), whitening only the drift fit preserved it (PCA-10 0.79–0.96), and whitening only the OT cost degraded it (PCA-10 0.53–0.79; PCA-20 0.28–0.43)",
     ["0.37", "0.58", "0.79", "0.96", "0.53", "0.79", "0.28", "0.43"], "Legend Fig 3", "(PCA-20 0.37–0.58), whitening only the drift fit (PCA-10 0.79–0.96), whitening only the OT cost (PCA-10 0.53–0.79, PCA-20 0.28–0.43)",
     ["0.37", "0.58", "0.79", "0.96", "0.53", "0.79", "0.28", "0.43"]),
    ("J_ref relative Frobenius error", "Results 3", "(1.000–1.005)", ["1.000", "1.005"], "Legend Fig 3", "error is 1.000–1.005", ["1.000", "1.005"]),
    ("J_ref norm ratio", "Results 3", "2–7 % of the reference's", ["0.02", "0.07"], "Legend Fig 3", "norm ratio 0.02–0.07", ["0.02", "0.07"]),
    ("J_ref per-cell cosine", "Results 3", "(−0.002 to 0.008)", ["−0.002", "0.008"], "Legend Fig 3", "median per-cell cosine −0.002 to 0.008", ["−0.002", "0.008"]),
    ("J_ref λ1 Spearman range", "Results 3", "is −0.107 to −0.005 across all four datasets (−0.018 to −0.005 on the three transition seeds", ["−0.107", "−0.005", "−0.018"], "Legend Fig 3", "per-cell λ₁ Spearman −0.107 to −0.005 across all four datasets (−0.018 to −0.005 on the three transition seeds)", ["−0.107", "−0.005", "−0.018"]),
    ("J_ref FD cosine", "Results 3", "median per-cell cosine ≈ 1.000 against", ["1.000"], "Legend Fig 3", "(median per-cell cosine ≈ 1.000)", ["1.000"]),
    ("transition R and offsets", "Results 3", "(1.91–2.69) from all no-forks (0.58–1.44), with timing offsets of 0, 1 and 5 bins", ["1.91", "2.69", "0.58", "1.44", "0", "1", "5"],
     "Legend Fig 3", "(1.91–2.69; |detected − true bin| 1, 0, 5) and three no-fork seeds (0.58–1.44)", ["1.91", "2.69", "0.58", "1.44", "0", "1", "5"]),
    ("S14b/S14c counts", "Results 3", "only 2 of 9 assessed seeds were eligible", ["2", "9"], "Legend Fig 3", "2 of 9 assessed seeds eligible", ["2", "9"]),
    ("S14c counts", "Results 3", "14 seeds stopped at the memory guard during generation and none of the 10 assessed was eligible", ["14", "10"],
     "Legend Fig 3", "14 seeds stopped at the generation memory guard; none of the 10 assessed was eligible", ["14", "10"]),
    ("minimum bins", "Results 3", "minimum bins of 42–46 cells", ["42", "46"], "Legend Fig 3", "(minimum bins 42–46 cells)", ["42", "46"]),
    ("dyngen fork AUC", "Results 4", "1.00, 0.92–0.99 and 0.31–0.38 (inverted)", ["1.00", "0.92", "0.99", "0.31", "0.38"], "Legend Fig 4", "flow J 1.00, 0.92–0.99 and 0.31–0.38 (inverted)", ["1.00", "0.92", "0.99", "0.31", "0.38"]),
    ("dyngen score AUC", "Results 4", "The score operator reached 0.98, 0.58 and 0.85", ["0.98", "0.58", "0.85"], "Legend Fig 4", "score J 0.98, 0.58, 0.85", ["0.98", "0.58", "0.85"]),
    ("dyngen rotation s1+s2", "Results 4", "(0.19–0.40 vs 0.07–0.09)", ["0.19", "0.40", "0.07", "0.09"], "Legend Fig 4", "cycle s1 0.19–0.32 and s2 0.36–0.40 vs linear 0.07–0.09", ["0.19", "0.40", "0.07", "0.09"]),
    ("dyngen rotation s0", "Results 4", "(0.02 vs 0.08–0.10)", ["0.02", "0.08", "0.10"], "Legend Fig 4", "cycle s0 0.02 vs linear s0 0.08–0.10", ["0.02", "0.08", "0.10"]),
    ("fork cells T3/T8", "Results 4", "22, 0 and 0 cells at the branch point (against 329, 746 and 776", ["22", "0", "0", "329", "746", "776"],
     "Legend Fig 4", "s10 22 vs 329, s16 0 vs 746, s18 0 vs 776", ["22", "0", "0", "329", "746", "776"]),
    ("fork cells T3/T8 (S2)", "Results 4", "22, 0 and 0 cells at the branch point (against 329, 746 and 776", ["22", "0", "0", "329", "746", "776"],
     "Legend Fig S2", "22, 0, 0 (s10, s16, s18; F1 not evaluable for s16 and s18) vs 329, 746, 776", ["22", "0", "0", "329", "746", "776"]),
    ("early fraction T8", "Results 4", "0.00–0.06 on two of three datasets (1.00 on the third)", ["0.00", "0.06", "1.00"], "Legend Fig S2", "s10 0.02–0.06, s16 1.00, s18 0.00", ["0.00", "0.06", "1.00"]),
    ("LARRY velocity cosine", "Results 5", "(median cosine 0.69 over 1,321 cells", ["0.69", "1321"], "Legend Fig 5", "(n = 1,321 cells with a lineage velocity; median 0.69)", ["0.69", "1321"]),
    ("LARRY lineage Spearman", "Results 5", "Spearman 0.56, pooled over the 15 PCA coordinates of 554 cell–clone pairs; maximum over five permutation nulls 0.15", ["0.56", "554", "0.15"], "Legend Fig 5", "554 barcoded cell–clone pairs, between the displacement predicted by integrating the drift over the 48-h interval and the observed displacement to the clone's next-day centroid (◆), vs five permutation nulls (●; pre-registered): reference 0.56 (null max 0.15)", ["0.56", "554", "0.15"]),
    ("LARRY energy distance", "Results 5", "(energy distance 0.69 vs 0.87 and 0.88; post hoc 95 % bootstrap intervals 0.66–0.73, 0.85–0.89 and 0.83–0.94", ["0.69", "0.87", "0.88", "0.66", "0.73"],
     "Legend Fig 5", "reference flow 0.69, tuned flow 0.94; mean-displacement baseline 0.87, no-movement baseline 0.88. Error bars (reference, post hoc): 95 % bootstrap intervals over cells (200 resamples): flow 0.66–0.73", ["0.69", "0.87", "0.88", "0.66", "0.73"]),
    ("Abstract energy distance", "Abstract", "energy distance 0.69, 95 % CI 0.66–0.73, vs 0.87 and 0.88", ["0.69", "0.87", "0.88"], "Legend Fig 5", "reference flow 0.69, tuned flow 0.94; mean-displacement baseline 0.87, no-movement baseline 0.88", ["0.69", "0.87", "0.88"]),
    ("LARRY seed/estimator stability", "Results 5", "(λ₁ Spearman 0.92–0.95", ["0.92", "0.95"], "Legend Fig 5", "Dynamo SparseVFC 0.92–0.95", ["0.92", "0.95"]),
    ("LARRY OT vs lineage per cell", "Results 5", "(λ₁ Spearman 0.27, null maximum 0.27; leading-eigenvector |cos| 0.29, null maximum 0.32", ["0.27", "0.27", "0.29", "0.32"],
     "Legend Fig S4", "λ₁ Spearman 0.27 (null max 0.27), |cos| 0.29 (null max 0.32)", ["0.27", "0.27", "0.29", "0.32"]),
    ("S11 criteria", "Results 5", "met 6 of 14 criteria", ["6", "14"], "Legend Fig 5", "6 of 14 met", ["6", "14"]),
    ("S11 criteria (S4)", "Results 5", "met 6 of 14 criteria", ["6", "14"], "Legend Fig S4", "6 of 14 criteria met", ["6", "14"]),
    ("mini-batch agreement", "Results 5", "(median velocity cosine 0.9998; λ₁ Spearman 0.9999)", ["0.9998", "0.9999"], "Legend Fig 5", "0.9998 and λ₁ Spearman 0.9999", ["0.9998", "0.9999"]),
    ("full LARRY OT", "Results 5", "in 291 s (2.5 GB)", ["291", "2.5"], "Legend Fig 5", "OT 291 s (2.54 GB)", ["291", "2.54"]),
    ("full LARRY fit", "Results 5", "in 21 min (1.3 GB)", ["21", "1.3"], "Legend Fig 5", "1,278 s (1.31 GB)", ["21.3", "1.31"]),
    ("dense cost matrix", "Results 5", "(cost matrix ≈ 21 GB)", ["21"], "Legend Fig 5", "(cost matrix ≈ 21 GB)", ["21"]),
    ("cell-type Spearman", "Results 5", "(Spearman 0.82 over 11 cell types)", ["0.82", "11"], "Legend Fig 5", "11 cell types) vs the 4,000-per-day S11 result: Spearman 0.82", ["0.82", "11"]),
    ("low-rank cosine", "Results 5", "(velocity cosine 0.79–0.80", ["0.79", "0.80"], "Legend Fig 5", "low-rank cosine 0.79–0.80", ["0.79", "0.80"]),
    ("scNT velocity cosines", "Results 5", "labeling vs Dynamo −0.05, labeling vs scVelo 0.03, scVelo vs Dynamo 0.15", ["−0.05", "0.03", "0.15"],
     "Legend Fig 5", "labeling vs Dynamo −0.05, labeling vs scVelo 0.03, scVelo vs Dynamo 0.15", ["−0.05", "0.03", "0.15"]),
    ("score-J AUC presets", "Results 6", "(AUC 0.16–0.31 on default, 0.15–0.50 on stress, 0.17–0.29 on batch, with AUC < 0.45 on 8 of these 9 seeds, and 0.43–0.52 on transition",
     ["0.16", "0.31", "0.15", "0.50", "0.17", "0.29", "8", "9", "0.43", "0.52"], "Legend Fig 6",
     "default 0.16–0.31, stress 0.15–0.50 and batch 0.17–0.29 (AUC < 0.45, i.e. committed cells ranked above saddle cells, on 8 of 9 seeds), 0.43–0.52 on transition",
     ["0.16", "0.31", "0.15", "0.50", "0.17", "0.29", "8", "9", "0.43", "0.52"]),
    ("S18 counts", "Results 6", "(10,691 non-targeting controls; 865 of 1,299 targets", ["10691", "865", "1299"], "Legend Fig 6", "10,691 non-targeting controls; 865 of 1,299 targets", ["10691", "865", "1299"]),
    ("S18 capture", "Results 6", "a median 4.5 % of each knockdown's centroid shift, against 58.9 %", ["0.045", "0.589"], "Legend Fig 6", "local score-J soft axes 0.045, global score J 0.038, local covariance 0.292, global covariance 0.589", ["0.045", "0.589"]),
    ("S18 random expectation", "Results 6", "random-subspace expectation of 15 %", ["0.15"], "Legend Fig 6", "random-subspace expectation 0.15", ["0.15"]),
    ("S18 P1", "Results 6", "(≥ 70 % of targets above their random-null 95th percentile) reached 5.3 %", ["0.70", "0.053"], "Legend Fig 6", "≥ 0.70): 0.053", ["0.70", "0.053"]),
    ("S18b H1", "Results 6", "a median 31 % of each knockdown's energy distance (required ≥ 50 %)", ["0.31", "0.50"], "Legend Fig 6", "median 0.31 (required ≥ 0.5)", ["0.31", "0.5"]),
    ("S18b H2", "Results 6", "across 59 basins (partial Spearman ρ +0.05, one-sided p 0.65", ["59", "0.05", "0.65"], "Legend Fig 6", "across 59 basins; partial Spearman ρ (ranks residualized on basin size and covariance trace) 0.05, one-sided p (negative) 0.65", ["59", "0.05", "0.65"]),
    ("A1 operator view", "Results 7", "recall 0.72–0.79 at 3.4–4.1× their base rate", ["0.72", "0.79", "3.4", "4.1"], "Legend Fig 7", "recall 0.72–0.79, precision 3.4–4.1× base rate", ["0.72", "0.79", "3.4", "4.1"]),
    ("A1 descriptor view", "Results 7", "higher recall (0.88–0.95) but failed precision on one seed (1.5×)", ["0.88", "0.95", "1.5"], "Legend Fig 7", "recall 0.88–0.95, precision 1.5–3.2×", ["0.88", "0.95", "1.5"]),
    ("A2 NMI", "Results 7", "0.35–0.39 for flow archetypes, 0.35–0.37 for Leiden at matched K and 0.05–0.31 for score archetypes", ["0.35", "0.39", "0.35", "0.37", "0.05", "0.31"],
     "Legend Fig 7", "flow 0.35–0.39, Leiden 0.35–0.37, score 0.05–0.31", ["0.35", "0.39", "0.35", "0.37", "0.05", "0.31"]),
    ("A3 ARI", "Results 7", "0.22–0.44 in the operator view and −0.02 to 0.53 in the descriptor view", ["0.22", "0.44", "−0.02", "0.53"], "Legend Fig 7", "operator view 0.22–0.44, descriptor view −0.02 to 0.53", ["0.22", "0.44", "−0.02", "0.53"]),
    ("K per view", "Results 7", "K = 8), and by K-means on rank-transformed descriptors (descriptor view, K = 2)", ["8", "2"], "Legend Fig 7", "K = 8) and by K-means on rank-transformed magnitude-free descriptors (descriptor view, K = 2)", ["8", "2"]),
    ("S22 pooled/Undifferentiated rho", "Results 6", "The pooled correlation between clone λ₁ and fate entropy was ρ = −0.003; within undifferentiated cells it was −0.10", ["−0.003", "−0.10"],
     "Legend Fig 6", "λ₁ ρ −0.003 (one-sided permutation p 1); Undifferentiated (1,401 pairs): λ₁ ρ −0.10", ["−0.003", "−0.10"]),
    ("S22 baselines and partial", "Results 6", "(|ρ| 0.22 vs 0.10 for λ₁), and the partial correlation given local density was −0.08", ["0.22", "0.10", "−0.08"],
     "Legend Fig 6", "λ₁ ρ −0.10 (p 1), centroid distance 0.22, density 0.08; partial ρ of λ₁ given density −0.08", ["0.22", "0.10", "−0.08"]),  # |λ₁ ρ| compared (text quotes |ρ|)  # |λ₁ ρ| compared (text quotes |ρ|)
    ("S18d RPE1 n_T", "Results 6", "underpowered (1 qualifying target; ≥ 10 required)", ["1", "10"], "Legend Fig 6", "underpowered: 1 qualifying T target, ≥ 10 required", ["1", "10"]),
    ("diffusion 48 h", "Discussion", "count noise in the embedding exceeds 48-h diffusion", ["48"], "Legend Fig S1", "exceeds the 48-h effective diffusion", ["48"]),
]
def agree(a, b):
    na, nb = ndec(a), ndec(b); nd = min(na, nb)
    return abs(num(a) - num(b)) <= 0.5 * 10 ** (-nd) + 1e-12
TLR = []
for q_, ts, tq, tv, ls, lq, lv in TL:
    assert tq in SEC[ts], (ts, tq); assert lq in SEC[ls], (ls, lq); assert len(tv) == len(lv), q_
    ok = all(agree(a, b) for a, b in zip(tv, lv))
    TLR.append(dict(quantity=q_, text_section=ts, text_quote=tq, legend=ls, legend_quote=lq,
                    text_values=", ".join(tv), legend_values=", ".join(lv), status="AGREE" if ok else "DISAGREE"))
TLR = pd.DataFrame(TLR)
print(TLR.status.value_counts().to_dict())
print(TLR[TLR.status == "DISAGREE"].to_string())

{'AGREE': 57}
Empty DataFrame
Columns: [quantity, text_section, text_quote, legend, legend_quote, text_values, legend_values, status]
Index: []


## Write NUMBERS_v57.md and numbers_v57.csv

In [18]:
tab = pd.DataFrame(ROWS)
counts = {k: int(v) for k, v in tab.status.value_counts().to_dict().items()}
prov.value("status_counts", counts)
prov.value("n_uncovered_tokens", int(len(UNCOV)))
prov.value("n_consistency_problems", len(PROBLEMS))
prov.value("text_vs_legend", TLR.status.value_counts().to_dict())
out_csv = FIGDIR / "numbers_v57.csv"; tab.to_csv(out_csv, index=False)
def esc(s):
    return str(s).replace("|", "\\|").replace("\n", " ")
lines = ["# NUMBERS_v57 — provenance of every number in the final v57 manuscript", "",
         f"Generated by `notebooks/numbers_v57.ipynb` (do not edit by hand). Manuscript: `{MS_PATH}` (final; not edited).",
         f"Manuscript sha256: `{sha256(resolve(MS_PATH))}`.", "",
         "Scope: every numeric claim in the Abstract, Changes from v56, Results, Discussion, Methods, figure legends (Figs 1–7, Supplementary "
         "Figs S1–S4) and the Supplementary tables list. The scJDO v1 (v56) analyses are not part of v57; statements that cite v56 by version are "
         "not v57 numbers and are not checked. Comparison rules: a quoted decimal is matched after rounding the recomputed value to the quoted number of decimals "
         "(half away from zero); a quoted range is matched against the recomputed min–max under the stated filter; \"≈ 0\" means |x| < 0.1 for every row; "
         "\"≈ 1.0\" means every value rounds to 1.0; \"under X\" / \"≤ X\" are inequalities. Values stated in committed result reports (RESULT_*.md) count as "
         "recomputed. Configuration constants, numbers that exist only in review notes and numbers from bundles not imported (S18d, S22) are NOT-RECOMPUTABLE. "
         "S15 rows are read from the merged final S15 results (`benchmarks/synthetic_multimodal/s15/`).", "",
         f"Summary: {counts.get('MATCH', 0)} MATCH, {counts.get('MISMATCH', 0)} MISMATCH, {counts.get('NOT-RECOMPUTABLE', 0)} NOT-RECOMPUTABLE ({len(tab)} rows). Coverage audit: {len(UNCOV)} numeric tokens outside checked snippets.", ""]
mm = tab[tab.status == "MISMATCH"]
lines += ["## MISMATCH rows", "", "| # | section | quoted text | recomputed | source file | column / filter | task | note |", "|---|---|---|---|---|---|---|---|"]
for i, r in mm.iterrows():
    lines.append(f"| {i + 1} | {r.section} | {esc(r.quoted)} | {esc(r.recomputed)} | `{esc(r.source)}` | {esc(r['filter'])} | {r.task} | {esc(r.note)} |")
lines += ["", "## Internal consistency of the final manuscript", "",
          f"References: {CIT['n_refs']} entries, numbered consecutively: {CIT['refs_consecutive']}; cited but missing: {CIT['cited_missing'] or 'none'}; "
          f"never cited: {CIT['uncited'] or 'none'}. Reference numbers first cited before lower-numbered references (numbering not in order of first appearance): "
          + ("; ".join(f"[{a}] is first cited before {', '.join(f'[{x}]' for x in b)}" for a, b in CIT["out_of_order_first_citations"]) or "none") + ".", "",
          "### Callout and citation problems", "", "| # | check | item | detail |", "|---|---|---|---|"]
for i, (c_, it_, d_) in enumerate(PROBLEMS):
    lines.append(f"| {i + 1} | {c_} | {esc(it_)} | {esc(d_)} |")
lines += ["", "### Numbers in both main text and a legend", "",
          f"{(TLR.status == 'AGREE').sum()} AGREE, {(TLR.status == 'DISAGREE').sum()} DISAGREE (values compared at the coarser quoted precision; % converted to fractions; legend 1,278 s compared as 21.3 min).", "",
          "| quantity | text section | text quote | text values | legend | legend quote | legend values | status |", "|---|---|---|---|---|---|---|---|"]
for _, r in TLR.sort_values("status").iterrows():
    lines.append(f"| {esc(r.quantity)} | {r.text_section} | {esc(r.text_quote)} | {esc(r.text_values)} | {r.legend} | {esc(r.legend_quote)} | {esc(r.legend_values)} | **{r.status}** |")
lines += ["", "## Coverage audit", "", "Numeric tokens in the manuscript sections that are not inside a checked snippet and not an identifier / label "
          "(seed and dataset IDs, gate and task labels, dates, accessions, dimensions in embedding names, notation):", ""]
lines += [f"- {r.section}: `{r.token}` — …{esc(r.context)}…" for _, r in UNCOV.iterrows()] or ["- none"]
lines += ["", "## All rows", "", "| # | section | quoted text | recomputed | source file | column / filter | task | status | note |", "|---|---|---|---|---|---|---|---|---|"]
for i, r in tab.iterrows():
    lines.append(f"| {i + 1} | {r.section} | {esc(r.quoted)} | {esc(r.recomputed)} | `{esc(r.source)}` | {esc(r['filter'])} | {r.task} | **{r.status}** | {esc(r.note)} |")
lines += ["", "## Source files (sha256 at the time of this run)", ""]
for k, (p, t) in sorted(F.items(), key=lambda kv: kv[1][0]):
    lines.append(f"- `{p}` ({t}): `{sha256(resolve(p))}`")
(REPO / "manuscript" / "NUMBERS_v57.md").write_text("\n".join(lines) + "\n")
prov.panel("table", "NUMBERS_v57.md and manuscript/figures/numbers_v57.csv (final manuscript)")
prov.write()
print(counts)
print(mm[["section", "quoted", "recomputed", "source"]].to_string())

{'MATCH': 308, 'NOT-RECOMPUTABLE': 25}
Empty DataFrame
Columns: [section, quoted, recomputed, source]
Index: []
